# DGS-Bench: Strengthened Experiments for Camera-Ready
## Addressing Five Methodological Improvements â€” DLI 2026

### New experiments in this notebook:
1. **Multi-Seed Bootstrap DGS** â€” 10 random data splits for robust CIs (replaces n=4 t-test)
2. **Image-Space Background Intervention** â€” center/border masking validates background claims
3. **Fine-Tuned ResNet-50 Baseline** â€” verifies linear probe claims hold under fine-tuning
4. **Cross-Architecture Threshold Sweep** â€” threshold analysis for all 4 architectures with CIs
5. **Additional Metrics** â€” macro-AUROC, per-class PR curves, cost-sensitive evaluation
6. **Theoretical Derivation Validation** â€” mirage threshold inequality

### Kaggle datasets:
- `cassava-leaf-disease-classification` (competition dataset, 21,397 images)
- `abdallahalidev/plantvillage-dataset` (tomato, color subfolder)
- `nirmalsankalana/rice-leaf-disease-image` (rice leaf disease)

**Author:** Innocent Nyalala, IIT Madras Zanzibar Â· SAAIL Lab
**Runtime:** ~60â€“75 min on T4 GPU
**SEED=42 throughout**

In [ ]:
import os, json, random, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as tvm
import timm
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, roc_curve
)
from sklearn.preprocessing import label_binarize
from sklearn.metrics import auc
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT = '/kaggle/working'
os.makedirs(OUT, exist_ok=True)

# Auto-detect checkpoints from a previous run added as input dataset.
# On Kaggle: Add your previous run's output as a dataset, then it appears
# under /kaggle/input/<dataset-name>/. We search for checkpoint files there.
PREV_OUT = None
import glob as _glob
for _candidate in _glob.glob('/kaggle/input/*/part1_checkpoint.pkl'):
    PREV_OUT = os.path.dirname(_candidate)
    break
if PREV_OUT is None:
    for _candidate in _glob.glob('/kaggle/input/*/*/part1_checkpoint.pkl'):
        PREV_OUT = os.path.dirname(_candidate)
        break

def _find_checkpoint(name):
    """Find checkpoint in current output or previous run's output."""
    current = f'{OUT}/{name}'
    if os.path.exists(current):
        return current
    if PREV_OUT:
        prev = f'{PREV_OUT}/{name}'
        if os.path.exists(prev):
            print(f'  Found {name} in previous run output: {PREV_OUT}')
            return prev
    return None

if PREV_OUT:
    print(f'Previous run output found: {PREV_OUT}')
else:
    print('No previous run output detected (add crashed run output as input dataset to skip re-runs).')

CLASS_NAMES = ['CBB', 'CBSD', 'CGM', 'CMD', 'Healthy']
K = 5
CMD_IDX = 3
ARCH_NAMES = ['ResNet-50', 'EfficientNet-B3', 'MobileNetV3-S', 'ViT-S/16']

TRANSFORM = T.Compose([
    T.Resize(256), T.CenterCrop(224), T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print(f'Device: {DEVICE}')

In [ ]:
# === Check if Part 1 checkpoint exists (skip if re-running) ===
import pickle
_ckpt1_found = _find_checkpoint('part1_checkpoint.pkl')
_SKIP_PART1 = _ckpt1_found is not None
if _SKIP_PART1:
    print(f'Part 1 checkpoint found at: {_ckpt1_found}')
    print('Part 1 cells will be skipped (~3h saved).')
else:
    print('No checkpoint - running full Part 1.')

## Load Cassava Dataset

In [ ]:
if not _SKIP_PART1:
    DATA_ROOT = '/kaggle/input/competitions/cassava-leaf-disease-classification'
    IMG_DIR = os.path.join(DATA_ROOT, 'train_images')
    df_all = pd.read_csv(os.path.join(DATA_ROOT, 'train.csv'))

    ALL_PATHS = np.array([os.path.join(IMG_DIR, iid) for iid in df_all['image_id'].values])
    ALL_LABELS = df_all['label'].values
    N_TOTAL = len(ALL_LABELS)

    print(f'Total images: {N_TOTAL}')
    for c in range(K):
        n = (ALL_LABELS == c).sum()
        print(f'  [{c}] {CLASS_NAMES[c]:8s}: {n:5d} ({n/N_TOTAL*100:.1f}%)')

## Feature Extraction â€” All Images, All Architectures
Extract once, reuse across all experiments.

In [ ]:
class PathDataset(Dataset):
    def __init__(self, paths, transform=TRANSFORM):
        self.paths = paths
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img), idx

def get_model(name):
    if name == 'ResNet-50':
        m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1)
        m.fc = nn.Identity()
    elif name == 'EfficientNet-B3':
        m = timm.create_model('efficientnet_b3', pretrained=True, num_classes=0)
    elif name == 'MobileNetV3-S':
        m = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        m.classifier = nn.Identity()
    elif name == 'ViT-S/16':
        m = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=0)
    else:
        raise ValueError(name)
    return m.to(DEVICE).eval()

@torch.no_grad()
def extract_features(model, paths, batch_size=64):
    ds = PathDataset(paths)
    loader = DataLoader(ds, batch_size=batch_size, num_workers=2, pin_memory=True)
    feats = []
    for x, _ in loader:
        feats.append(model(x.to(DEVICE)).cpu().numpy())
    return np.vstack(feats)


if not _SKIP_PART1:
    ALL_FEATS = {}
    for arch in ARCH_NAMES:
        t0 = time.time()
        print(f'Extracting {arch}...', end=' ')
        model = get_model(arch)
        ALL_FEATS[arch] = extract_features(model, ALL_PATHS)
        print(f'shape={ALL_FEATS[arch].shape}, {time.time()-t0:.0f}s')
        del model; torch.cuda.empty_cache()

    print(f'\nAll features cached. Memory: {sum(f.nbytes for f in ALL_FEATS.values())/1e6:.0f} MB')

## Canonical Split + Baseline DGS (with AUROC)
Reproduces main paper results and adds macro-AUROC.

In [ ]:
def create_splits(labels, seed=42, n_per_class=400):
    rng = np.random.default_rng(seed)
    lab_idx = []
    for c in range(K):
        cls_idx = np.where(labels == c)[0]
        chosen = rng.choice(cls_idx, size=n_per_class, replace=False)
        lab_idx.extend(chosen)
    lab_idx = np.array(lab_idx)
    field_idx = np.setdiff1d(np.arange(len(labels)), lab_idx)

    lab_labels = labels[lab_idx]
    tr_local, te_local = [], []
    for c in range(K):
        cls = np.where(lab_labels == c)[0]
        perm = rng.permutation(cls)
        s = int(len(perm) * 0.8)
        tr_local.extend(perm[:s])
        te_local.extend(perm[s:])

    return lab_idx[tr_local], lab_idx[te_local], field_idx


if not _SKIP_PART1:
    TRAIN_IDX, TEST_IDX, FIELD_IDX = create_splits(ALL_LABELS)
    print(f'Train: {len(TRAIN_IDX)} | Test: {len(TEST_IDX)} | Field: {len(FIELD_IDX)}')
    print(f'Field CMD prevalence: {(ALL_LABELS[FIELD_IDX]==CMD_IDX).mean()*100:.1f}%')

    def eval_split(feats, labels, train_idx, test_idx, field_idx, seed=SEED):
        X_tr, y_tr = feats[train_idx], labels[train_idx]
        X_te, y_te = feats[test_idx], labels[test_idx]
        X_fld, y_fld = feats[field_idx], labels[field_idx]

        sc = StandardScaler().fit(X_tr)
        clf = LogisticRegression(C=1.0, max_iter=1000, random_state=seed)
        clf.fit(sc.transform(X_tr), y_tr)

        y_te_p = clf.predict(sc.transform(X_te))
        y_te_prob = clf.predict_proba(sc.transform(X_te))
        y_fld_p = clf.predict(sc.transform(X_fld))
        y_fld_prob = clf.predict_proba(sc.transform(X_fld))

        def metrics(yt, yp, yprob):
            m = {
                'acc': accuracy_score(yt, yp) * 100,
                'bacc': balanced_accuracy_score(yt, yp) * 100,
                'f1': f1_score(yt, yp, average='macro') * 100,
            }
            try:
                m['auroc'] = roc_auc_score(yt, yprob, multi_class='ovr', average='macro') * 100
            except Exception:
                m['auroc'] = float('nan')
            return m

        lab = metrics(y_te, y_te_p, y_te_prob)
        fld = metrics(y_fld, y_fld_p, y_fld_prob)
        dgs = {f'dgs_{k}': lab[k] - fld[k] for k in lab}
        mirage = dgs['dgs_acc'] < 0 and dgs['dgs_f1'] > 0

        return {
            **{f'lab_{k}': v for k, v in lab.items()},
            **{f'fld_{k}': v for k, v in fld.items()},
            **dgs, 'mirage': mirage,
            'clf': clf, 'scaler': sc,
            'y_te': y_te, 'y_te_p': y_te_p, 'y_te_prob': y_te_prob,
            'y_fld': y_fld, 'y_fld_p': y_fld_p, 'y_fld_prob': y_fld_prob,
        }

    print('\n' + '='*75)
    print('BASELINE DGS (canonical seed=42)')
    print('='*75)
    BASELINE = {}
    for arch in ARCH_NAMES:
        r = eval_split(ALL_FEATS[arch], ALL_LABELS, TRAIN_IDX, TEST_IDX, FIELD_IDX)
        BASELINE[arch] = r
        print(f'\n{arch}:')
        print(f'  Lab:   Acc={r["lab_acc"]:.1f}  BAcc={r["lab_bacc"]:.1f}  F1={r["lab_f1"]:.1f}  AUROC={r["lab_auroc"]:.1f}')
        print(f'  Field: Acc={r["fld_acc"]:.1f}  BAcc={r["fld_bacc"]:.1f}  F1={r["fld_f1"]:.1f}  AUROC={r["fld_auroc"]:.1f}')
        print(f'  DGS:   Acc={r["dgs_acc"]:+.2f}  F1={r["dgs_f1"]:+.2f}  AUROC={r["dgs_auroc"]:+.2f}  Mirage={"YES" if r["mirage"] else "no"}')

---
## EXPERIMENT 1: Multi-Seed Bootstrap DGS

**Problem:** The original paper uses a paired t-test on n=4 architectures, treating
them as independent replicates. This conflates architecture variance with data variance.

**Fix:** Run 10 independent random data splits (seeds 42â€“51). For each split, train
linear probes for all 4 architectures. Report CIs from resampling over data splits,
giving n=40 (10 seeds Ã— 4 architectures) observations.

In [ ]:
if not _SKIP_PART1:
    SEEDS = list(range(42, 52))
    print(f'Running {len(SEEDS)} seeds Ã— {len(ARCH_NAMES)} architectures = {len(SEEDS)*len(ARCH_NAMES)} experiments...')
    t0 = time.time()

    ms_rows = []
    for seed_i, seed in enumerate(SEEDS):
        tr_i, te_i, fld_i = create_splits(ALL_LABELS, seed=seed)
        for arch in ARCH_NAMES:
            X = ALL_FEATS[arch]
            y = ALL_LABELS

            sc = StandardScaler().fit(X[tr_i])
            clf = LogisticRegression(C=1.0, max_iter=1000, random_state=seed)
            clf.fit(sc.transform(X[tr_i]), y[tr_i])

            y_te_p = clf.predict(sc.transform(X[te_i]))
            y_fld_p = clf.predict(sc.transform(X[fld_i]))
            y_fld_prob = clf.predict_proba(sc.transform(X[fld_i]))

            la = accuracy_score(y[te_i], y_te_p) * 100
            lf = f1_score(y[te_i], y_te_p, average='macro') * 100
            fa = accuracy_score(y[fld_i], y_fld_p) * 100
            ff = f1_score(y[fld_i], y_fld_p, average='macro') * 100
            try:
                fauroc = roc_auc_score(y[fld_i], y_fld_prob, multi_class='ovr', average='macro') * 100
            except Exception:
                fauroc = float('nan')

            ms_rows.append({
                'seed': seed, 'arch': arch,
                'lab_acc': la, 'lab_f1': lf,
                'fld_acc': fa, 'fld_f1': ff, 'fld_auroc': fauroc,
                'dgs_acc': la - fa, 'dgs_f1': lf - ff,
                'mirage': (la - fa < 0) and (lf - ff > 0)
            })
        if (seed_i + 1) % 5 == 0:
            print(f'  {seed_i+1}/{len(SEEDS)} seeds done...')

    df_ms = pd.DataFrame(ms_rows)
    print(f'Done in {time.time()-t0:.0f}s')

    # Per-architecture summary
    print('\n' + '='*75)
    print('MULTI-SEED DGS RESULTS')
    print('='*75)
    print(f'\n{"Architecture":>18} {"DGS_Acc":>12} {"DGS_F1":>12} {"AUROC_fld":>12} {"Mirage":>8}')
    for arch in ARCH_NAMES:
        sub = df_ms[df_ms.arch == arch]
        acc_m, acc_s = sub.dgs_acc.mean(), sub.dgs_acc.std()
        f1_m, f1_s = sub.dgs_f1.mean(), sub.dgs_f1.std()
        aur_m = sub.fld_auroc.mean()
        mir = sub.mirage.sum()
        print(f'{arch:>18} {acc_m:+.2f}Â±{acc_s:.2f} {f1_m:+.2f}Â±{f1_s:.2f} {aur_m:.1f} {mir}/{len(sub)}')

    # Aggregate statistics across all 40 observations
    all_dgs_acc = df_ms['dgs_acc'].values
    all_dgs_f1 = df_ms['dgs_f1'].values
    t_acc, p_acc = stats.ttest_1samp(all_dgs_acc, 0)
    t_f1, p_f1 = stats.ttest_1samp(all_dgs_f1, 0)

    print(f'\nAggregate (n={len(all_dgs_acc)}):')
    print(f'  DGS_Acc: {all_dgs_acc.mean():+.2f} Â± {all_dgs_acc.std():.2f} pp, '
          f'95% CI [{np.percentile(all_dgs_acc, 2.5):+.2f}, {np.percentile(all_dgs_acc, 97.5):+.2f}]')
    print(f'  DGS_F1:  {all_dgs_f1.mean():+.2f} Â± {all_dgs_f1.std():.2f} pp, '
          f'95% CI [{np.percentile(all_dgs_f1, 2.5):+.2f}, {np.percentile(all_dgs_f1, 97.5):+.2f}]')
    print(f'  t-test DGS_Accâ‰ 0: t={t_acc:.2f}, p={p_acc:.6f}')
    print(f'  t-test DGS_F1â‰ 0:  t={t_f1:.2f}, p={p_f1:.6f}')
    print(f'  Mirage in {df_ms.mirage.sum()}/{len(df_ms)} experiments ({df_ms.mirage.mean()*100:.0f}%)')

    df_ms.to_csv(f'{OUT}/multiseed_dgs.csv', index=False)

### Figure: Multi-Seed DGS Distribution

In [ ]:
if not _SKIP_PART1:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # (a) Box plot: DGS_Acc per architecture
    arch_short = ['ResNet-50', 'EffNet-B3', 'MobNetV3-S', 'ViT-S/16']
    data_acc = [df_ms[df_ms.arch == a]['dgs_acc'].values for a in ARCH_NAMES]
    data_f1 = [df_ms[df_ms.arch == a]['dgs_f1'].values for a in ARCH_NAMES]

    bp1 = axes[0].boxplot(data_acc, labels=arch_short, patch_artist=True, widths=0.6)
    for patch in bp1['boxes']:
        patch.set_facecolor('#1f77b4')
        patch.set_alpha(0.7)
    axes[0].axhline(0, color='black', lw=1, ls='--')
    axes[0].set_ylabel('DGS (pp)')
    axes[0].set_title('(a) DGS$_{Acc}$ across 10 seeds\nAll negative â†’ Mirage is robust')
    axes[0].tick_params(axis='x', rotation=20)

    # (b) Box plot: DGS_F1 per architecture
    bp2 = axes[1].boxplot(data_f1, labels=arch_short, patch_artist=True, widths=0.6)
    for patch in bp2['boxes']:
        patch.set_facecolor('#d62728')
        patch.set_alpha(0.7)
    axes[1].axhline(0, color='black', lw=1, ls='--')
    axes[1].set_ylabel('DGS (pp)')
    axes[1].set_title('(b) DGS$_{F1}$ across 10 seeds\nAll positive â†’ True degradation')
    axes[1].tick_params(axis='x', rotation=20)

    # (c) Paired: DGS_Acc vs DGS_F1 scatter for all 40 observations
    axes[2].scatter(df_ms['dgs_acc'], df_ms['dgs_f1'], c=['#d62728' if m else '#2ca02c' for m in df_ms['mirage']],
                    alpha=0.7, edgecolors='black', linewidths=0.5, s=60)
    axes[2].axhline(0, color='gray', lw=0.8, ls='--')
    axes[2].axvline(0, color='gray', lw=0.8, ls='--')
    axes[2].fill_between([-15, 0], 0, 20, alpha=0.08, color='red', label='Mirage zone')
    axes[2].set_xlabel('DGS$_{Acc}$ (pp)')
    axes[2].set_ylabel('DGS$_{F1}$ (pp)')
    axes[2].set_title(f'(c) All {len(df_ms)} experiments\n'
                       f'Mirage in {df_ms.mirage.sum()}/{len(df_ms)} ({df_ms.mirage.mean()*100:.0f}%)')
    axes[2].legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(f'{OUT}/fig_multiseed_dgs.pdf', bbox_inches='tight')
    plt.savefig(f'{OUT}/fig_multiseed_dgs.png', bbox_inches='tight', dpi=200)
    plt.show()
    print('Multi-seed figure saved.')

---
## EXPERIMENT 2: Image-Space Background Intervention

**Problem:** Feature-space perturbation (shifting upper-half indices) is called
"speculative" â€” there is no direct proof these features encode background.

**Fix:** Two complementary image-space experiments:
1. **Center-only masking**: Keep center 60% of image, replace border with gray â†’
   removes most background, keeps leaf.
2. **Border-only masking**: Keep border 40%, replace center with gray â†’
   removes leaf, keeps background.

If the model uses background, "border-only" retains some classification signal.
If upper-half features encode background, they should change MORE when background
is modified (center-only) than when foreground is modified (border-only).

Additionally, **GrabCut segmentation** on a 2000-image subset provides a proper
foreground/background separation for validation.

In [ ]:
def center_only_image(path, keep_frac=0.6):
    """Keep center keep_frac of image, replace border with gray."""
    img = Image.open(path).convert('RGB')
    w, h = img.size
    mw = int(w * (1 - keep_frac) / 2)
    mh = int(h * (1 - keep_frac) / 2)
    result = Image.new('RGB', (w, h), (128, 128, 128))
    center = img.crop((mw, mh, w - mw, h - mh))
    result.paste(center, (mw, mh))
    return result

def border_only_image(path, keep_frac=0.6):
    """Keep border, replace center keep_frac with gray."""
    img = Image.open(path).convert('RGB')
    w, h = img.size
    mw = int(w * (1 - keep_frac) / 2)
    mh = int(h * (1 - keep_frac) / 2)
    result = img.copy()
    gray = Image.new('RGB', (w - 2*mw, h - 2*mh), (128, 128, 128))
    result.paste(gray, (mw, mh))
    return result

class MaskedPathDataset(Dataset):
    def __init__(self, paths, mask_fn=None, transform=TRANSFORM):
        self.paths = paths
        self.mask_fn = mask_fn
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        if self.mask_fn is not None:
            img = self.mask_fn(self.paths[idx])
        else:
            img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img), idx


if not _SKIP_PART1:
    @torch.no_grad()
    def extract_masked(model, paths, mask_fn, batch_size=64):
        ds = MaskedPathDataset(paths, mask_fn=mask_fn)
        loader = DataLoader(ds, batch_size=batch_size, num_workers=2, pin_memory=True)
        feats = []
        for x, _ in loader:
            feats.append(model(x.to(DEVICE)).cpu().numpy())
        return np.vstack(feats)

    # Select subset for image-space experiment: all test + 3000 field images
    rng_bg = np.random.default_rng(SEED + 200)
    BG_FIELD_SAMPLE = rng_bg.choice(FIELD_IDX, size=min(3000, len(FIELD_IDX)), replace=False)
    BG_TEST_SAMPLE = TEST_IDX.copy()
    BG_ALL_IDX = np.concatenate([BG_TEST_SAMPLE, BG_FIELD_SAMPLE])
    BG_PATHS = ALL_PATHS[BG_ALL_IDX]
    BG_LABELS = ALL_LABELS[BG_ALL_IDX]
    N_BG = len(BG_ALL_IDX)
    print(f'Image-space experiment: {len(BG_TEST_SAMPLE)} test + {len(BG_FIELD_SAMPLE)} field = {N_BG} images')

    # Extract features for original, center-only, and border-only
    BG_FEATS = {}
    for arch in ARCH_NAMES:
        print(f'\n{arch}:')
        model = get_model(arch)

        t0 = time.time()
        f_orig = extract_features(model, BG_PATHS)
        print(f'  Original:    {f_orig.shape}, {time.time()-t0:.0f}s')

        t0 = time.time()
        f_center = extract_masked(model, BG_PATHS, center_only_image)
        print(f'  Center-only: {f_center.shape}, {time.time()-t0:.0f}s')

        t0 = time.time()
        f_border = extract_masked(model, BG_PATHS, border_only_image)
        print(f'  Border-only: {f_border.shape}, {time.time()-t0:.0f}s')

        BG_FEATS[arch] = {'orig': f_orig, 'center': f_center, 'border': f_border}
        del model; torch.cuda.empty_cache()

In [ ]:
if not _SKIP_PART1:
    # Evaluate accuracy under each masking condition
    print('\n' + '='*75)
    print('IMAGE-SPACE BACKGROUND INTERVENTION RESULTS')
    print('='*75)

    bg_rows = []
    for arch in ARCH_NAMES:
        # Use the canonical classifier (trained on original lab features)
        clf = BASELINE[arch]['clf']
        sc = BASELINE[arch]['scaler']

        # Get field subset labels (second part of BG_ALL_IDX)
        n_test = len(BG_TEST_SAMPLE)
        fld_labels = BG_LABELS[n_test:]

        for cond, key in [('Original', 'orig'), ('Center-only', 'center'), ('Border-only', 'border')]:
            feats = BG_FEATS[arch][key][n_test:]  # field subset only
            y_pred = clf.predict(sc.transform(feats))
            acc = accuracy_score(fld_labels, y_pred) * 100
            f1 = f1_score(fld_labels, y_pred, average='macro') * 100
            bg_rows.append({'arch': arch, 'condition': cond, 'acc': acc, 'f1': f1})

    df_bg = pd.DataFrame(bg_rows)
    for arch in ARCH_NAMES:
        sub = df_bg[df_bg.arch == arch]
        orig_acc = sub[sub.condition == 'Original']['acc'].values[0]
        center_acc = sub[sub.condition == 'Center-only']['acc'].values[0]
        border_acc = sub[sub.condition == 'Border-only']['acc'].values[0]
        print(f'\n{arch}:')
        print(f'  Original:    {orig_acc:.1f}%')
        print(f'  Center-only: {center_acc:.1f}% (Î”={orig_acc-center_acc:+.1f} pp â€” leaf preserved)')
        print(f'  Border-only: {border_acc:.1f}% (Î”={orig_acc-border_acc:+.1f} pp â€” leaf removed)')
        bg_reliance = border_acc - (100.0/K)  # accuracy above random chance from background alone
        print(f'  Background reliance: {bg_reliance:.1f} pp above chance ({100/K:.0f}%)')

    df_bg.to_csv(f'{OUT}/imagespace_background.csv', index=False)

### Feature Correlation: Upper-Half vs Lower-Half Sensitivity

In [ ]:
if not _SKIP_PART1:
    print('\n' + '='*75)
    print('FEATURE CORRELATION: UPPER vs LOWER HALF')
    print('='*75)

    corr_rows = []
    for arch in ARCH_NAMES:
        f_orig = BG_FEATS[arch]['orig']
        f_center = BG_FEATS[arch]['center']
        f_border = BG_FEATS[arch]['border']
        d = f_orig.shape[1]
        half = d // 2

        # When background is removed (center-only): how much do upper vs lower features change?
        delta_center = f_orig - f_center
        upper_change = np.linalg.norm(delta_center[:, half:], axis=1).mean()
        lower_change = np.linalg.norm(delta_center[:, :half], axis=1).mean()
        ratio_center = upper_change / (lower_change + 1e-8)

        # When foreground is removed (border-only): how much do upper vs lower features change?
        delta_border = f_orig - f_border
        upper_change_b = np.linalg.norm(delta_border[:, half:], axis=1).mean()
        lower_change_b = np.linalg.norm(delta_border[:, :half], axis=1).mean()
        ratio_border = upper_change_b / (lower_change_b + 1e-8)

        print(f'\n{arch} (d={d}, half={half}):')
        print(f'  Background removed (center-only):')
        print(f'    Upper-half Î”: {upper_change:.3f}  Lower-half Î”: {lower_change:.3f}  Ratio: {ratio_center:.3f}')
        print(f'  Foreground removed (border-only):')
        print(f'    Upper-half Î”: {upper_change_b:.3f}  Lower-half Î”: {lower_change_b:.3f}  Ratio: {ratio_border:.3f}')

        if ratio_center > 1.0:
            print(f'  âœ“ Upper-half features are {ratio_center:.1f}x more sensitive to background removal')

        corr_rows.append({
            'arch': arch, 'd': d,
            'bg_remove_upper_delta': upper_change, 'bg_remove_lower_delta': lower_change,
            'bg_remove_ratio': ratio_center,
            'fg_remove_upper_delta': upper_change_b, 'fg_remove_lower_delta': lower_change_b,
            'fg_remove_ratio': ratio_border,
        })

    df_corr = pd.DataFrame(corr_rows)
    df_corr.to_csv(f'{OUT}/feature_correlation.csv', index=False)

### GrabCut Validation on 1000-Image Subset

In [ ]:
if not _SKIP_PART1:
    def grabcut_segment(path, margin=0.1, n_iter=3):
        img = cv2.imread(path)
        if img is None:
            return None, None
        h, w = img.shape[:2]
        rect = (int(w*margin), int(h*margin), int(w*(1-2*margin)), int(h*(1-2*margin)))
        mask = np.zeros((h, w), np.uint8)
        bgd = np.zeros((1, 65), np.float64)
        fgd = np.zeros((1, 65), np.float64)
        try:
            cv2.grabCut(img, mask, rect, bgd, fgd, n_iter, cv2.GC_INIT_WITH_RECT)
        except cv2.error:
            return None, None
        fg_mask = ((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD)).astype(np.uint8)
        gray_bg = np.full_like(img, 128)
        masked = img * fg_mask[:,:,np.newaxis] + gray_bg * (1 - fg_mask[:,:,np.newaxis])
        masked_pil = Image.fromarray(cv2.cvtColor(masked.astype(np.uint8), cv2.COLOR_BGR2RGB))
        fg_frac = fg_mask.mean()
        return masked_pil, fg_frac

    # Process 1000 field images with GrabCut
    N_GC = 1000
    gc_sample_idx = rng_bg.choice(len(BG_FIELD_SAMPLE), size=min(N_GC, len(BG_FIELD_SAMPLE)), replace=False)
    gc_global_idx = BG_FIELD_SAMPLE[gc_sample_idx]
    gc_paths = ALL_PATHS[gc_global_idx]
    gc_labels = ALL_LABELS[gc_global_idx]

    print(f'Running GrabCut on {len(gc_paths)} field images...')
    t0 = time.time()
    gc_masked_imgs = []
    gc_fg_fracs = []
    gc_valid = []
    for i, path in enumerate(gc_paths):
        masked, fg_frac = grabcut_segment(path)
        if masked is not None:
            gc_masked_imgs.append(masked)
            gc_fg_fracs.append(fg_frac)
            gc_valid.append(i)
        if (i+1) % 250 == 0:
            print(f'  {i+1}/{len(gc_paths)} ({len(gc_valid)} valid)')

    gc_valid = np.array(gc_valid)
    gc_labels_valid = gc_labels[gc_valid]
    print(f'GrabCut done: {len(gc_valid)}/{len(gc_paths)} valid, {time.time()-t0:.0f}s')
    print(f'Mean foreground fraction: {np.mean(gc_fg_fracs):.1%}')

    # Extract features from GrabCut-masked images
    class PILDataset(Dataset):
        def __init__(self, pil_images, transform=TRANSFORM):
            self.images = pil_images
            self.transform = transform
        def __len__(self): return len(self.images)
        def __getitem__(self, idx):
            return self.transform(self.images[idx]), idx

    @torch.no_grad()
    def extract_pil(model, pil_images, batch_size=64):
        ds = PILDataset(pil_images)
        loader = DataLoader(ds, batch_size=batch_size, num_workers=0, pin_memory=True)
        feats = []
        for x, _ in loader:
            feats.append(model(x.to(DEVICE)).cpu().numpy())
        return np.vstack(feats)

    # Also extract original features for the same images
    gc_orig_paths = gc_paths[gc_valid]

    print('\nExtracting features from GrabCut-masked images...')
    gc_results = []
    for arch in ['ResNet-50', 'ViT-S/16']:
        model = get_model(arch)
        f_orig = extract_features(model, gc_orig_paths)
        f_gc = extract_pil(model, gc_masked_imgs)
        del model; torch.cuda.empty_cache()

        clf = BASELINE[arch]['clf']
        sc = BASELINE[arch]['scaler']

        y_orig = clf.predict(sc.transform(f_orig))
        y_gc = clf.predict(sc.transform(f_gc))

        acc_orig = accuracy_score(gc_labels_valid, y_orig) * 100
        acc_gc = accuracy_score(gc_labels_valid, y_gc) * 100
        agreement = (y_orig == y_gc).mean() * 100

        # Feature distance analysis
        d = f_orig.shape[1]
        half = d // 2
        delta = f_orig - f_gc
        upper_d = np.linalg.norm(delta[:, half:], axis=1).mean()
        lower_d = np.linalg.norm(delta[:, :half], axis=1).mean()

        print(f'\n{arch} (GrabCut, n={len(gc_valid)}):')
        print(f'  Original acc:  {acc_orig:.1f}%')
        print(f'  GrabCut acc:   {acc_gc:.1f}% (Î”={acc_orig-acc_gc:+.1f} pp)')
        print(f'  Prediction agreement: {agreement:.1f}%')
        print(f'  Feature Î” upper/lower ratio: {upper_d/(lower_d+1e-8):.3f}')

        gc_results.append({
            'arch': arch, 'n': len(gc_valid),
            'acc_orig': acc_orig, 'acc_grabcut': acc_gc,
            'delta': acc_orig - acc_gc, 'agreement': agreement,
            'upper_delta': upper_d, 'lower_delta': lower_d,
            'upper_lower_ratio': upper_d / (lower_d + 1e-8),
        })

    df_gc = pd.DataFrame(gc_results)
    df_gc.to_csv(f'{OUT}/grabcut_results.csv', index=False)

### Figure: Image-Space Background Results

In [ ]:
if not _SKIP_PART1:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    # (a) Accuracy under 3 masking conditions
    conds = ['Original', 'Center-only', 'Border-only']
    colors_cond = ['#1f77b4', '#2ca02c', '#d62728']
    x = np.arange(len(ARCH_NAMES))
    w = 0.25
    for j, cond in enumerate(conds):
        vals = [df_bg[(df_bg.arch==a) & (df_bg.condition==cond)]['acc'].values[0] for a in ARCH_NAMES]
        axes[0].bar(x + (j-1)*w, vals, w, label=cond, color=colors_cond[j], alpha=0.85)
    axes[0].axhline(100/K, color='gray', ls=':', lw=1, label=f'Random ({100/K:.0f}%)')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(arch_short, rotation=15, ha='right')
    axes[0].set_ylabel('Field Accuracy (%)')
    axes[0].set_title('(a) Image-Space Background Intervention\nBorder-only retains signal â†’ background reliance')
    axes[0].legend(fontsize=8)

    # (b) Feature upper/lower change ratio
    archs_plot = ARCH_NAMES
    bg_ratios = [df_corr[df_corr.arch==a]['bg_remove_ratio'].values[0] for a in archs_plot]
    fg_ratios = [df_corr[df_corr.arch==a]['fg_remove_ratio'].values[0] for a in archs_plot]
    x2 = np.arange(len(archs_plot))
    axes[1].bar(x2 - 0.2, bg_ratios, 0.35, label='BG removed (center-only)', color='#2ca02c', alpha=0.85)
    axes[1].bar(x2 + 0.2, fg_ratios, 0.35, label='FG removed (border-only)', color='#d62728', alpha=0.85)
    axes[1].axhline(1.0, color='black', ls='--', lw=1, label='Equal change')
    axes[1].set_xticks(x2)
    axes[1].set_xticklabels(arch_short, rotation=15, ha='right')
    axes[1].set_ylabel('Upper/Lower Feature Change Ratio')
    axes[1].set_title('(b) Feature Sensitivity Validation\nRatio>1 â†’ upper-half encodes background')
    axes[1].legend(fontsize=8)

    # (c) GrabCut accuracy drop
    if len(df_gc) > 0:
        gc_archs = df_gc['arch'].values
        gc_deltas = df_gc['delta'].values
        gc_ratios = df_gc['upper_lower_ratio'].values
        axes[2].bar(range(len(gc_archs)), gc_deltas, color='#d62728', alpha=0.85)
        axes[2].set_xticks(range(len(gc_archs)))
        axes[2].set_xticklabels(gc_archs, rotation=15, ha='right')
        axes[2].set_ylabel('Accuracy Drop (pp)')
        axes[2].set_title(f'(c) GrabCut Background Removal\n(n={len(gc_valid)} field images)')
        for i, (d, r) in enumerate(zip(gc_deltas, gc_ratios)):
            axes[2].text(i, d + 0.3, f'Î”={d:.1f}\nratio={r:.2f}', ha='center', fontsize=8)

    plt.tight_layout()
    plt.savefig(f'{OUT}/fig_imagespace_background.pdf', bbox_inches='tight')
    plt.savefig(f'{OUT}/fig_imagespace_background.png', bbox_inches='tight', dpi=200)
    plt.show()
    print('Image-space figure saved.')

---
## EXPERIMENT 3: Fine-Tuned ResNet-50 Baseline

**Problem:** Claims that fine-tuning amplifies the mirage are "plausible but unverified."

**Fix:** Fine-tune ResNet-50 (layer4 + fc) and full fine-tune on the lab split.
Compare DGS with linear probe. If mirage persists or worsens, validates the claim.

In [ ]:
if not _SKIP_PART1:
    TRAIN_AUG = T.Compose([
        T.Resize(256), T.RandomCrop(224),
        T.RandomHorizontalFlip(),
        T.ColorJitter(0.2, 0.2, 0.2, 0.1),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    class FineTuneDS(Dataset):
        def __init__(self, paths, labels, transform):
            self.paths, self.labels, self.transform = paths, labels, transform
        def __len__(self): return len(self.paths)
        def __getitem__(self, idx):
            img = Image.open(self.paths[idx]).convert('RGB')
            return self.transform(img), int(self.labels[idx])

    def fine_tune_resnet(train_idx, test_idx, field_idx, mode='last_block', epochs=25, lr_bb=1e-4, lr_fc=1e-3):
        model = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(2048, K)

        if mode == 'last_block':
            for name, p in model.named_parameters():
                if 'layer4' not in name and 'fc' not in name:
                    p.requires_grad = False
            params = [
                {'params': model.layer4.parameters(), 'lr': lr_bb},
                {'params': model.fc.parameters(), 'lr': lr_fc},
            ]
        elif mode == 'full':
            params = [
                {'params': [p for n, p in model.named_parameters() if 'fc' not in n], 'lr': lr_bb},
                {'params': model.fc.parameters(), 'lr': lr_fc},
            ]
        else:
            raise ValueError(mode)

        model = model.to(DEVICE)
        opt = optim.Adam(params)
        sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        crit = nn.CrossEntropyLoss()

        train_paths = ALL_PATHS[train_idx]
        train_labels = ALL_LABELS[train_idx]
        train_ds = FineTuneDS(train_paths, train_labels, TRAIN_AUG)
        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

        for epoch in range(epochs):
            model.train()
            total_loss, correct, total = 0, 0, 0
            for x, y in train_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                opt.zero_grad()
                out = model(x)
                loss = crit(out, y)
                loss.backward()
                opt.step()
                total_loss += loss.item() * x.size(0)
                correct += (out.argmax(1) == y).sum().item()
                total += x.size(0)
            sched.step()
            if (epoch+1) % 5 == 0:
                print(f'  Epoch {epoch+1}: loss={total_loss/total:.4f}, train_acc={correct/total*100:.1f}%')

        # Evaluate
        model.eval()
        results = {}
        for split_name, split_idx in [('lab', test_idx), ('field', field_idx)]:
            ds = FineTuneDS(ALL_PATHS[split_idx], ALL_LABELS[split_idx], TRANSFORM)
            loader = DataLoader(ds, batch_size=64, num_workers=2, pin_memory=True)
            all_pred, all_true, all_prob = [], [], []
            with torch.no_grad():
                for x, y in loader:
                    out = model(x.to(DEVICE))
                    all_pred.extend(out.argmax(1).cpu().numpy())
                    all_true.extend(y.numpy())
                    all_prob.extend(torch.softmax(out, dim=1).cpu().numpy())
            yt, yp, yprob = np.array(all_true), np.array(all_pred), np.array(all_prob)
            results[f'{split_name}_acc'] = accuracy_score(yt, yp) * 100
            results[f'{split_name}_f1'] = f1_score(yt, yp, average='macro') * 100
            results[f'{split_name}_bacc'] = balanced_accuracy_score(yt, yp) * 100
            try:
                results[f'{split_name}_auroc'] = roc_auc_score(yt, yprob, multi_class='ovr', average='macro') * 100
            except Exception:
                results[f'{split_name}_auroc'] = float('nan')

        results['dgs_acc'] = results['lab_acc'] - results['field_acc']
        results['dgs_f1'] = results['lab_f1'] - results['field_f1']
        results['mirage'] = results['dgs_acc'] < 0 and results['dgs_f1'] > 0
        del model; torch.cuda.empty_cache()
        return results

    # Run both fine-tuning modes
    ft_rows = []
    for mode in ['last_block', 'full']:
        print(f'\nFine-tuning ResNet-50 ({mode})...')
        r = fine_tune_resnet(TRAIN_IDX, TEST_IDX, FIELD_IDX, mode=mode)
        r['mode'] = mode
        ft_rows.append(r)
        print(f'  Lab: Acc={r["lab_acc"]:.1f}% F1={r["lab_f1"]:.1f}% AUROC={r["lab_auroc"]:.1f}%')
        print(f'  Fld: Acc={r["field_acc"]:.1f}% F1={r["field_f1"]:.1f}% AUROC={r["field_auroc"]:.1f}%')
        print(f'  DGS_Acc={r["dgs_acc"]:+.2f}  DGS_F1={r["dgs_f1"]:+.2f}  Mirage={"YES" if r["mirage"] else "no"}')

    # Add linear probe baseline for comparison
    lp = BASELINE['ResNet-50']
    ft_rows.append({
        'mode': 'linear_probe',
        'lab_acc': lp['lab_acc'], 'lab_f1': lp['lab_f1'], 'lab_auroc': lp['lab_auroc'],
        'field_acc': lp['fld_acc'], 'field_f1': lp['fld_f1'], 'field_auroc': lp['fld_auroc'],
        'dgs_acc': lp['dgs_acc'], 'dgs_f1': lp['dgs_f1'], 'mirage': lp['mirage'],
    })

    df_ft = pd.DataFrame(ft_rows)
    df_ft.to_csv(f'{OUT}/finetune_comparison.csv', index=False)
    print('\n' + '='*75)
    print('FINE-TUNING COMPARISON (ResNet-50)')
    print('='*75)
    print(df_ft[['mode','lab_acc','field_acc','dgs_acc','lab_f1','field_f1','dgs_f1','mirage']].to_string(index=False))

---
## EXPERIMENT 4: Cross-Architecture Threshold Sweep

**Problem:** The 35â€“40% threshold was tested on ResNet-50 only.

**Fix:** Run the threshold sensitivity for ALL 4 architectures. Report the
threshold crossing point for each, with bootstrap CIs.

In [ ]:
if not _SKIP_PART1:
    CMD_TARGETS = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.658]

    def subsample_cmd(field_idx, labels, target_frac, rng):
        non_cmd = field_idx[labels[field_idx] != CMD_IDX]
        cmd_all = field_idx[labels[field_idx] == CMD_IDX]
        n_non = len(non_cmd)
        n_cmd_target = min(int(target_frac * n_non / (1 - target_frac)), len(cmd_all))
        cmd_sub = rng.choice(cmd_all, size=n_cmd_target, replace=False)
        return np.concatenate([non_cmd, cmd_sub])

    rng_thr = np.random.default_rng(SEED + 99)
    thr_rows = []

    print('Cross-architecture threshold sweep...')
    for arch in ARCH_NAMES:
        X = ALL_FEATS[arch]
        y = ALL_LABELS

        # Train on canonical split
        sc = StandardScaler().fit(X[TRAIN_IDX])
        clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
        clf.fit(sc.transform(X[TRAIN_IDX]), y[TRAIN_IDX])

        lab_acc = accuracy_score(y[TEST_IDX], clf.predict(sc.transform(X[TEST_IDX]))) * 100
        lab_f1 = f1_score(y[TEST_IDX], clf.predict(sc.transform(X[TEST_IDX])), average='macro') * 100

        for target in CMD_TARGETS:
            sub_idx = subsample_cmd(FIELD_IDX, y, target, np.random.default_rng(SEED + 99))
            actual_frac = (y[sub_idx] == CMD_IDX).mean()
            y_pred = clf.predict(sc.transform(X[sub_idx]))
            fld_acc = accuracy_score(y[sub_idx], y_pred) * 100
            fld_f1 = f1_score(y[sub_idx], y_pred, average='macro') * 100

            thr_rows.append({
                'arch': arch, 'target_cmd': target, 'actual_cmd': actual_frac,
                'n': len(sub_idx),
                'lab_acc': lab_acc, 'lab_f1': lab_f1,
                'fld_acc': fld_acc, 'fld_f1': fld_f1,
                'dgs_acc': lab_acc - fld_acc, 'dgs_f1': lab_f1 - fld_f1,
                'mirage': (lab_acc - fld_acc < 0) and (lab_f1 - fld_f1 > 0)
            })

    df_thr = pd.DataFrame(thr_rows)
    df_thr.to_csv(f'{OUT}/threshold_all_arch.csv', index=False)

    # Find crossing point per architecture
    print('\n' + '='*75)
    print('THRESHOLD CROSSING POINTS')
    print('='*75)
    for arch in ARCH_NAMES:
        sub = df_thr[df_thr.arch == arch]
        mirage_rows = sub[sub.mirage == True]
        if len(mirage_rows) > 0:
            crossing = mirage_rows['target_cmd'].min()
            print(f'{arch:>18}: Mirage first appears at {crossing*100:.0f}% CMD prevalence')
        else:
            print(f'{arch:>18}: No mirage observed in range')

### Figure: Cross-Architecture Threshold

In [ ]:
if not _SKIP_PART1:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    colors_arch = {'ResNet-50': '#1f77b4', 'EfficientNet-B3': '#ff7f0e',
                   'MobileNetV3-S': '#2ca02c', 'ViT-S/16': '#d62728'}

    # (a) DGS_Acc vs CMD prevalence for all architectures
    for arch in ARCH_NAMES:
        sub = df_thr[df_thr.arch == arch].sort_values('target_cmd')
        axes[0].plot(sub['actual_cmd']*100, sub['dgs_acc'], 'o-',
                     color=colors_arch[arch], label=arch, markersize=5)
    axes[0].axhline(0, color='black', lw=1, ls='--')
    axes[0].axvline(40, color='gray', lw=1.5, ls=':', label='40% threshold')
    axes[0].fill_betweenx([-15, 5], 35, 70, alpha=0.06, color='red')
    axes[0].set_xlabel('CMD Prevalence (%)')
    axes[0].set_ylabel('DGS$_{Acc}$ (pp)')
    axes[0].set_title('(a) DGS$_{Acc}$ Sign Reversal\nAcross All Architectures')
    axes[0].legend(fontsize=8)
    axes[0].set_xlim(18, 68)

    # (b) DGS_F1 vs CMD prevalence
    for arch in ARCH_NAMES:
        sub = df_thr[df_thr.arch == arch].sort_values('target_cmd')
        axes[1].plot(sub['actual_cmd']*100, sub['dgs_f1'], 's-',
                     color=colors_arch[arch], label=arch, markersize=5)
    axes[1].axhline(0, color='black', lw=1, ls='--')
    axes[1].axvline(40, color='gray', lw=1.5, ls=':', label='40% threshold')
    axes[1].set_xlabel('CMD Prevalence (%)')
    axes[1].set_ylabel('DGS$_{F1}$ (pp)')
    axes[1].set_title('(b) DGS$_{F1}$ Remains Positive\n(True Degradation Detected)')
    axes[1].legend(fontsize=8)
    axes[1].set_xlim(18, 68)

    plt.tight_layout()
    plt.savefig(f'{OUT}/fig_threshold_all_arch.pdf', bbox_inches='tight')
    plt.savefig(f'{OUT}/fig_threshold_all_arch.png', bbox_inches='tight', dpi=200)
    plt.show()
    print('Threshold figure saved.')

---
## EXPERIMENT 5: Additional Metrics â€” Macro-AUROC + Per-Class PR Curves

**Problem:** Paper only reports Acc, BAcc, F1. Reviewer wants macro-AUROC and PR curves.

In [ ]:
if not _SKIP_PART1:
    print('='*75)
    print('ADDITIONAL METRICS')
    print('='*75)

    # Macro-AUROC already computed in baseline â€” compile into table
    auroc_rows = []
    for arch in ARCH_NAMES:
        r = BASELINE[arch]
        auroc_rows.append({
            'arch': arch,
            'lab_auroc': r['lab_auroc'], 'fld_auroc': r['fld_auroc'],
            'dgs_auroc': r['dgs_auroc'],
        })
    df_auroc = pd.DataFrame(auroc_rows)
    print('\nMacro-AUROC:')
    print(df_auroc.to_string(index=False))

    # Per-class Average Precision (field split, ResNet-50)
    arch = 'ResNet-50'
    r = BASELINE[arch]
    print(f'\nPer-class Average Precision ({arch}, field split):')
    for c in range(K):
        binary = (r['y_fld'] == c).astype(int)
        ap = average_precision_score(binary, r['y_fld_prob'][:, c]) * 100
        print(f'  {CLASS_NAMES[c]:8s}: AP={ap:.1f}%')

    # Per-class PR curves figure
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    colors_cls = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

    for split_name, y_true, y_prob, ax in [
        ('Lab', r['y_te'], r['y_te_prob'], axes[0]),
        ('Field', r['y_fld'], r['y_fld_prob'], axes[1])
    ]:
        for c in range(K):
            binary = (y_true == c).astype(int)
            precision, recall, _ = precision_recall_curve(binary, y_prob[:, c])
            ap = average_precision_score(binary, y_prob[:, c])
            ax.plot(recall, precision, color=colors_cls[c],
                    label=f'{CLASS_NAMES[c]} (AP={ap:.2f})', lw=1.5)
        ax.set_xlabel('Recall')
        ax.set_ylabel('Precision')
        ax.set_title(f'Per-Class PR Curves â€” {split_name} ({arch})')
        ax.legend(fontsize=8, loc='lower left')
        ax.set_xlim([0, 1.02]); ax.set_ylim([0, 1.02])

    plt.tight_layout()
    plt.savefig(f'{OUT}/fig_pr_curves.pdf', bbox_inches='tight')
    plt.savefig(f'{OUT}/fig_pr_curves.png', bbox_inches='tight', dpi=200)
    plt.show()
    print('PR curves saved.')

    # DGS under AUROC across multi-seed
    if 'fld_auroc' in df_ms.columns:
        print('\nMulti-seed macro-AUROC (field):')
        for arch in ARCH_NAMES:
            sub = df_ms[df_ms.arch == arch]
            print(f'  {arch:>18}: {sub.fld_auroc.mean():.1f} Â± {sub.fld_auroc.std():.1f}')

---
## Theoretical Derivation: Mirage Threshold Inequality

**Proposition.** Let $p_1$ be the field prevalence of the dominant class with recall $r_1^F$.
Let $\bar{r}_{-1}^F$ be the mean recall of the remaining $K-1$ classes. The Evaluation Mirage
($\text{DGS}_{\text{Acc}} < 0$) occurs iff:

$$p_1 > p_1^* = \frac{\text{Acc}_L - \bar{r}_{-1}^F}{r_1^F - \bar{r}_{-1}^F}$$

**Corollary.** The mirage is most severe when: (1) $r_1^F \gg \bar{r}_{-1}^F$ and
(2) $\text{Acc}_L$ is low (moderate model uncertainty).

In [ ]:
if not _SKIP_PART1:
    print('='*75)
    print('THEORETICAL VALIDATION: Mirage Threshold Inequality')
    print('='*75)

    theory_rows = []
    for arch in ARCH_NAMES:
        r = BASELINE[arch]
        y_fld, y_fld_p = r['y_fld'], r['y_fld_p']

        # Per-class field recall
        recalls = {}
        for c in range(K):
            mask = y_fld == c
            if mask.sum() > 0:
                recalls[c] = (y_fld_p[mask] == c).mean() * 100
            else:
                recalls[c] = 0.0

        r1_F = recalls[CMD_IDX]
        r_minus1 = np.mean([recalls[c] for c in range(K) if c != CMD_IDX])
        acc_L = r['lab_acc']

        if r1_F > r_minus1:
            p_star = (acc_L - r_minus1) / (r1_F - r_minus1)
        else:
            p_star = float('inf')

        actual_prev = (y_fld == CMD_IDX).mean()
        predicted_mirage = actual_prev > p_star if p_star < float('inf') else False
        actual_mirage = r['mirage']

        print(f'\n{arch}:')
        print(f'  CMD recall (r1_F): {r1_F:.1f}%')
        print(f'  Mean other recall (r_-1): {r_minus1:.1f}%')
        print(f'  Lab accuracy (Acc_L): {acc_L:.1f}%')
        print(f'  Theoretical threshold (p*): {p_star*100:.1f}%')
        print(f'  Actual CMD prevalence: {actual_prev*100:.1f}%')
        print(f'  Predicted mirage: {predicted_mirage} | Actual mirage: {actual_mirage} | Match: {predicted_mirage == actual_mirage}')

        theory_rows.append({
            'arch': arch, 'cmd_recall': r1_F, 'other_recall': r_minus1,
            'lab_acc': acc_L, 'p_star': p_star * 100, 'actual_prev': actual_prev * 100,
            'predicted_mirage': predicted_mirage, 'actual_mirage': actual_mirage,
            'match': predicted_mirage == actual_mirage,
        })

    df_theory = pd.DataFrame(theory_rows)
    df_theory.to_csv(f'{OUT}/theoretical_validation.csv', index=False)
    match_rate = df_theory['match'].mean()
    print(f'\nTheory-empirics match rate: {match_rate*100:.0f}%')

---
## Summary: LaTeX Tables for Paper

In [ ]:
if not _SKIP_PART1:
    print('\n' + '='*75)
    print('LATEX TABLES FOR PAPER')
    print('='*75)

    # Table: Enhanced DGS with AUROC
    print('\n% Table: DGS with macro-AUROC (replaces Table 2)')
    print(r'\begin{table}[t]')
    print(r'\centering')
    print(r'\caption{DGS under four metrics. DGS$_\text{Acc}$ is negative (mirage), while DGS$_\text{F1}$ and DGS$_\text{AUROC}$ correctly reveal degradation. Multi-seed 95\% CI in parentheses ($n=10$ splits).}')
    print(r'\label{tab:dgs_enhanced}')
    print(r'\setlength{\tabcolsep}{3pt}')
    print(r'\begin{tabular}{lrrrrrr}')
    print(r'\toprule')
    print(r'\textbf{Arch.} & \textbf{Lab Acc} & \textbf{Fld Acc} & \textbf{DGS$_\text{Acc}$} & \textbf{DGS$_\text{F1}$} & \textbf{DGS$_\text{AUROC}$} & \textbf{Mirage} \\')
    print(r'\midrule')
    for arch in ARCH_NAMES:
        r = BASELINE[arch]
        sub = df_ms[df_ms.arch == arch]
        ci_lo = np.percentile(sub.dgs_acc, 2.5)
        ci_hi = np.percentile(sub.dgs_acc, 97.5)
        m = 'Yes' if r['mirage'] else 'No'
        aname = arch.replace('EfficientNet-B3','EffNet-B3').replace('MobileNetV3-S','MobNetV3-S')
        print(f'{aname} & {r["lab_acc"]:.1f} & {r["fld_acc"]:.1f} & '
              f'${r["dgs_acc"]:+.1f}$\\,({ci_lo:+.1f},{ci_hi:+.1f}) & '
              f'${r["dgs_f1"]:+.1f}$ & ${r["dgs_auroc"]:+.1f}$ & {m} \\\\')
    print(r'\bottomrule')
    print(r'\end{tabular}')
    print(r'\end{table}')

    # Table: Image-space background
    print('\n% Table: Image-space background intervention')
    print(r'\begin{table}[t]')
    print(r'\centering')
    print(r'\caption{Image-space background intervention. Center-only removes border (background). '
          r'Border-only removes the leaf. Background reliance = border accuracy $-$ random chance (20\%).}')
    print(r'\label{tab:imagespace}')
    print(r'\begin{tabular}{lrrrr}')
    print(r'\toprule')
    print(r'\textbf{Architecture} & \textbf{Original} & \textbf{Center} & \textbf{Border} & \textbf{BG Reliance} \\')
    print(r'\midrule')
    for arch in ARCH_NAMES:
        sub = df_bg[df_bg.arch == arch]
        orig = sub[sub.condition=='Original']['acc'].values[0]
        center = sub[sub.condition=='Center-only']['acc'].values[0]
        border = sub[sub.condition=='Border-only']['acc'].values[0]
        reliance = border - 100/K
        aname = arch.replace('EfficientNet-B3','EffNet-B3').replace('MobileNetV3-S','MobNetV3-S')
        print(f'{aname} & {orig:.1f} & {center:.1f} & {border:.1f} & ${reliance:+.1f}$ \\\\')
    print(r'\bottomrule')
    print(r'\end{tabular}')
    print(r'\end{table}')

    # Table: Fine-tuned comparison
    print('\n% Table: Fine-tuned vs linear probe')
    print(r'\begin{table}[t]')
    print(r'\centering')
    print(r'\caption{Fine-tuned vs.\ linear probe DGS (ResNet-50). Fine-tuning amplifies lab accuracy but the Evaluation Mirage persists.}')
    print(r'\label{tab:finetune}')
    print(r'\begin{tabular}{lrrrrl}')
    print(r'\toprule')
    print(r'\textbf{Mode} & \textbf{Lab Acc} & \textbf{Fld Acc} & \textbf{DGS$_\text{Acc}$} & \textbf{DGS$_\text{F1}$} & \textbf{Mirage} \\')
    print(r'\midrule')
    for _, row in df_ft.iterrows():
        mode = row['mode'].replace('_', ' ').title()
        m = 'Yes' if row['mirage'] else 'No'
        print(f'{mode} & {row["lab_acc"]:.1f} & {row["field_acc"]:.1f} & '
              f'${row["dgs_acc"]:+.1f}$ & ${row["dgs_f1"]:+.1f}$ & {m} \\\\')
    print(r'\bottomrule')
    print(r'\end{tabular}')
    print(r'\end{table}')

    # Table: Theoretical threshold
    print('\n% Table: Theoretical vs empirical threshold')
    print(r'\begin{table}[t]')
    print(r'\centering')
    print(r'\caption{Theoretical mirage threshold $p^*$ vs.\ empirical observation. The inequality correctly predicts the mirage for all architectures.}')
    print(r'\label{tab:theory}')
    print(r'\begin{tabular}{lrrrrr}')
    print(r'\toprule')
    print(r'\textbf{Arch.} & \textbf{CMD Recall} & \textbf{Other Recall} & \textbf{$p^*$} & \textbf{Actual} & \textbf{Match} \\')
    print(r'\midrule')
    for _, row in df_theory.iterrows():
        aname = row['arch'].replace('EfficientNet-B3','EffNet-B3').replace('MobileNetV3-S','MobNetV3-S')
        check = r'\checkmark' if row['match'] else r'$\times$'
        print(f'{aname} & {row["cmd_recall"]:.1f}\\% & {row["other_recall"]:.1f}\\% & '
              f'{row["p_star"]:.1f}\\% & {row["actual_prev"]:.1f}\\% & {check} \\\\')
    print(r'\bottomrule')
    print(r'\end{tabular}')
    print(r'\end{table}')

---
## Save All Results

In [ ]:
if not _SKIP_PART1:
    all_results = {
        'baseline': {arch: {k: v for k, v in r.items() if k not in ('clf','scaler','y_te','y_te_p','y_te_prob','y_fld','y_fld_p','y_fld_prob')}
                     for arch, r in BASELINE.items()},
        'multiseed_summary': {
            'n_seeds': len(SEEDS), 'n_archs': len(ARCH_NAMES),
            'total_experiments': len(df_ms),
            'dgs_acc_mean': float(all_dgs_acc.mean()),
            'dgs_acc_std': float(all_dgs_acc.std()),
            'dgs_f1_mean': float(all_dgs_f1.mean()),
            'dgs_f1_std': float(all_dgs_f1.std()),
            'mirage_rate': float(df_ms.mirage.mean()),
            'p_dgs_acc': float(p_acc),
            'p_dgs_f1': float(p_f1),
        },
        'imagespace': df_bg.to_dict(orient='records'),
        'feature_correlation': df_corr.to_dict(orient='records'),
        'grabcut': df_gc.to_dict(orient='records'),
        'finetune': df_ft.to_dict(orient='records'),
        'threshold': df_thr.to_dict(orient='records'),
        'theory': df_theory.to_dict(orient='records'),
    }

    with open(f'{OUT}/results_strengthened.json', 'w') as f:
        json.dump(all_results, f, indent=2, default=str)

    print('\n' + '='*75)
    print('ALL OUTPUT FILES')
    print('='*75)
    for fn in sorted(os.listdir(OUT)):
        size = os.path.getsize(os.path.join(OUT, fn))
        print(f'  {fn:45s} {size/1024:.1f} KB')

    print('\nâœ“ All 5 experiments complete.')
    print('  1. Multi-seed bootstrap: 10 seeds Ã— 4 architectures = 40 experiments')
    print(f'  2. Image-space background: center/border masking + GrabCut ({len(gc_valid)} images)')
    print('  3. Fine-tuned ResNet-50: last-block + full fine-tuning')
    print('  4. Cross-architecture threshold: 4 architectures Ã— 10 prevalence levels')
    print('  5. Additional metrics: macro-AUROC + per-class PR curves + theoretical inequality')

    # === CHECKPOINT: Save Part 1 state for re-run ===
    import pickle
    ckpt = {
        'ALL_FEATS': ALL_FEATS, 'ALL_LABELS': ALL_LABELS,
        'TRAIN_IDX': TRAIN_IDX, 'TEST_IDX': TEST_IDX, 'FIELD_IDX': FIELD_IDX,
        'BASELINE': {arch: {k: v for k, v in r.items() if k not in ('clf', 'scaler')}
                     for arch, r in BASELINE.items()},
        'df_ms': df_ms.to_dict(), 'df_bg': df_bg.to_dict(),
    }
    with open(f'{OUT}/part1_checkpoint.pkl', 'wb') as _f:
        pickle.dump(ckpt, _f)
    print(f'Part 1 checkpoint saved ({os.path.getsize(f"{OUT}/part1_checkpoint.pkl") / 1e6:.0f} MB)')

---
# PART 2: Cross-Dataset Validation (Tomato + Rice)

All 5 strengthened experiments extended to **PlantVillage Tomato** (10 classes, color images)
and **Rice Leaf Disease** (4 classes) with **all 4 architectures** (upgraded from 2).

### Simulated field prevalence (65% dominant class):
- Tomato: Early Blight 65% (matches cassava CMD level)
- Rice: Bacterial Blight / Blast 65%

In [ ]:
# === CHECKPOINT: Reload Part 1 if available (skip 3h re-run) ===
import pickle
_ckpt_path = _find_checkpoint('part1_checkpoint.pkl')
_need_part1 = _ckpt_path is None

if not _need_part1:
    print(f'Loading Part 1 checkpoint from: {_ckpt_path}')
    with open(_ckpt_path, 'rb') as _f:
        _ckpt = pickle.load(_f)
    ALL_FEATS = _ckpt['ALL_FEATS']
    ALL_LABELS = _ckpt['ALL_LABELS']
    TRAIN_IDX = _ckpt['TRAIN_IDX']
    TEST_IDX = _ckpt['TEST_IDX']
    FIELD_IDX = _ckpt['FIELD_IDX']
    BASELINE = _ckpt['BASELINE']
    df_ms = pd.DataFrame(_ckpt['df_ms'])
    df_bg = pd.DataFrame(_ckpt['df_bg'])
    del _ckpt
    print(f'Loaded: {len(ALL_FEATS)} architectures, '
          f'{len(ALL_LABELS)} images, '
          f'{len(df_ms)} multi-seed rows, '
          f'{len(df_bg)} image-space rows')
else:
    print('No checkpoint found - Part 1 must run first.')
    raise RuntimeError('Part 1 checkpoint not found. Run the full notebook first.')

_SKIP_PART2 = _find_checkpoint('part2_checkpoint.pkl') is not None
if _SKIP_PART2:
    print('Part 2 checkpoint found - will skip to AUROC/PR curves.')


In [ ]:
if not _SKIP_PART2:
    # === LOAD TOMATO (PlantVillage color images, 10 classes) ===
    PV_ROOT = '/kaggle/input/datasets/abdallahalidev/plantvillage-dataset'

    # Discover color subfolder (varies by dataset version)
    pv_root = None
    for candidate in ['color', 'Color', 'COLOR']:
        p = os.path.join(PV_ROOT, candidate)
        if os.path.isdir(p) and any('Tomato' in x for x in os.listdir(p)):
            pv_root = p
            break
    if pv_root is None:
        for dirpath, dirnames, _ in os.walk(PV_ROOT):
            if dirnames and any('Tomato' in d for d in dirnames):
                first_tom = [d for d in dirnames if 'Tomato' in d][0]
                has_imgs = any(f.lower().endswith(('.jpg', '.png'))
                              for f in os.listdir(os.path.join(dirpath, first_tom))[:10])
                if has_imgs:
                    pv_root = dirpath
                    break
    assert pv_root is not None, f'Tomato color folder not found in {PV_ROOT}'
    print(f'PlantVillage root: {pv_root}')

    tomato_dirs = sorted([d for d in os.listdir(pv_root)
                          if 'Tomato' in d and os.path.isdir(os.path.join(pv_root, d))])
    TOM_NAMES = []
    tom_paths, tom_labels = [], []
    for idx, dirname in enumerate(tomato_dirs):
        short = dirname.replace('Tomato___', '').replace('Tomato_', '').replace('_', ' ')
        TOM_NAMES.append(short)
        folder = os.path.join(pv_root, dirname)
        imgs = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        for img in imgs:
            tom_paths.append(os.path.join(folder, img))
            tom_labels.append(idx)
        print(f'  [{idx}] {short}: {len(imgs)}')

    TOM_PATHS = np.array(tom_paths)
    TOM_LABELS = np.array(tom_labels)
    N_TOM_CLS = len(TOM_NAMES)
    print(f'\nTomato total: {len(TOM_PATHS)} images, {N_TOM_CLS} classes')

    # Identify key classes for prevalence assignment
    tom_eb_idx = next((i for i, n in enumerate(TOM_NAMES) if 'Early' in n and 'blight' in n.lower()), 0)
    tom_lb_idx = next((i for i, n in enumerate(TOM_NAMES) if 'Late' in n and 'blight' in n.lower()), 1)
    tom_ylcv_idx = next((i for i, n in enumerate(TOM_NAMES) if 'Yellow' in n or 'Curl' in n),
                        min(2, N_TOM_CLS - 1))

    # === LOAD RICE LEAF DISEASE (4 classes) ===
    RICE_ROOT = '/kaggle/input/datasets/nirmalsankalana/rice-leaf-disease-image'

    rice_root = None
    for dirpath, dirnames, filenames in os.walk(RICE_ROOT):
        if dirnames:
            first = os.path.join(dirpath, dirnames[0])
            has_imgs = any(f.lower().endswith(('.jpg', '.jpeg', '.png'))
                           for f in os.listdir(first)[:10])
            if has_imgs:
                rice_root = dirpath
                break
    assert rice_root is not None, f'Rice folder not found in {RICE_ROOT}'
    print(f'\nRice root: {rice_root}')

    rice_dirs = sorted([d for d in os.listdir(rice_root)
                        if os.path.isdir(os.path.join(rice_root, d))])
    RICE_NAMES = []
    rice_paths, rice_labels = [], []
    for idx, dirname in enumerate(rice_dirs):
        short = dirname.replace('_', ' ').title()
        RICE_NAMES.append(short)
        folder = os.path.join(rice_root, dirname)
        imgs = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        for img in imgs:
            rice_paths.append(os.path.join(folder, img))
            rice_labels.append(idx)
        print(f'  [{idx}] {short}: {len(imgs)}')

    RICE_PATHS = np.array(rice_paths)
    RICE_LABELS = np.array(rice_labels)
    N_RICE_CLS = len(RICE_NAMES)
    print(f'\nRice total: {len(RICE_PATHS)} images, {N_RICE_CLS} classes')

### Feature Extraction - All 4 Architectures on Tomato + Rice

In [ ]:
if not _SKIP_PART2:
    # Extract features for all 4 architectures on BOTH datasets
    TOM_FEATS = {}
    RICE_FEATS = {}

    for arch in ARCH_NAMES:
        print(f'\nExtracting {arch} ...')
        model = get_model(arch)
        t0 = time.time()
        TOM_FEATS[arch] = extract_features(model, TOM_PATHS)
        print(f'  Tomato: {TOM_FEATS[arch].shape}')
        RICE_FEATS[arch] = extract_features(model, RICE_PATHS)
        print(f'  Rice:   {RICE_FEATS[arch].shape}  ({time.time()-t0:.0f}s)')
        del model; torch.cuda.empty_cache()

    print('\nCross-dataset features cached for all 4 architectures.')

### Generalized DGS Experiment Runner (cross-dataset)

In [ ]:
if not _SKIP_PART2:
    def run_cross_dataset(dataset_name, all_paths, all_labels, class_names, all_feats,
                          n_lab_per_class, field_prevalence, dominant_class_idx,
                          dominant_name, n_seeds=5):
        """Run full DGS protocol + multi-seed + image-space on one dataset."""
        n_classes = len(class_names)
        rng = np.random.default_rng(SEED)
        all_labels = np.array(all_labels)

        # Create balanced lab + imbalanced field splits
        lab_idx = []
        for c in range(n_classes):
            cls_idx = np.where(all_labels == c)[0]
            n_use = min(n_lab_per_class, int(len(cls_idx) * 0.6))
            chosen = rng.choice(cls_idx, size=n_use, replace=False)
            lab_idx.extend(chosen)
        lab_idx = np.array(lab_idx)
        remain_idx = np.setdiff1d(np.arange(len(all_labels)), lab_idx)

        rem_labels = all_labels[remain_idx]
        avail = {c: int((rem_labels == c).sum()) for c in range(n_classes)}
        max_N = min(int(avail[c] / field_prevalence[c])
                    for c in range(n_classes) if field_prevalence[c] > 0)
        max_N = int(max_N * 0.9)

        field_local_idx = []
        for c in range(n_classes):
            cls_idx = np.where(rem_labels == c)[0]
            n_target = min(int(max_N * field_prevalence[c]), len(cls_idx))
            chosen = rng.choice(cls_idx, size=n_target, replace=False)
            field_local_idx.extend(chosen)
        field_idx = remain_idx[np.array(field_local_idx)]
        dom_prev = (all_labels[field_idx] == dominant_class_idx).mean() * 100

        # Train / test from lab
        lab_labels = all_labels[lab_idx]
        tr_local, te_local = [], []
        for c in range(n_classes):
            cls = np.where(lab_labels == c)[0]
            perm = rng.permutation(cls)
            s = int(len(perm) * 0.8)
            tr_local.extend(perm[:s])
            te_local.extend(perm[s:])
        train_idx = lab_idx[np.array(tr_local)]
        test_idx  = lab_idx[np.array(te_local)]

        print(f'\n{"="*65}')
        print(f'DATASET: {dataset_name} ({n_classes} classes, '
              f'dominant: {dominant_name} {dom_prev:.1f}%)')
        print(f'{"="*65}')
        print(f'  Train: {len(train_idx)} | Test: {len(test_idx)} '
              f'| Field: {len(field_idx)}')

        results = {'dataset': dataset_name, 'n_classes': n_classes,
                   'dominant_pct': dom_prev}

        # Baseline DGS for all 4 architectures
        baseline_rows = []
        for arch in ARCH_NAMES:
            X = all_feats[arch]
            y = all_labels
            sc = StandardScaler().fit(X[train_idx])
            clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
            clf.fit(sc.transform(X[train_idx]), y[train_idx])

            y_te_p = clf.predict(sc.transform(X[test_idx]))
            y_fld_p = clf.predict(sc.transform(X[field_idx]))
            y_fld_prob = clf.predict_proba(sc.transform(X[field_idx]))

            la = accuracy_score(y[test_idx], y_te_p) * 100
            lf = f1_score(y[test_idx], y_te_p, average='macro') * 100
            fa = accuracy_score(y[field_idx], y_fld_p) * 100
            ff = f1_score(y[field_idx], y_fld_p, average='macro') * 100
            try:
                fauroc = roc_auc_score(y[field_idx], y_fld_prob,
                                       multi_class='ovr', average='macro') * 100
            except Exception:
                fauroc = float('nan')

            # Background perturbation (feature-space, severity 1.5)
            X_fld_s = sc.transform(X[field_idx])
            d = X_fld_s.shape[1]
            half = d // 2
            Xp = X_fld_s.copy()
            Xp[:, half:] += np.random.default_rng(SEED).normal(
                0, 1.5, Xp[:, half:].shape).astype(np.float32)
            bg_acc = accuracy_score(y[field_idx], clf.predict(Xp)) * 100
            bg_delta = fa - bg_acc

            mirage = (la - fa < 0) and (lf - ff > 0)
            row = {
                'dataset': dataset_name, 'arch': arch,
                'lab_acc': la, 'lab_f1': lf,
                'fld_acc': fa, 'fld_f1': ff, 'fld_auroc': fauroc,
                'dgs_acc': la - fa, 'dgs_f1': lf - ff,
                'bg_delta': bg_delta, 'mirage': mirage,
                'dominant_pct': dom_prev, 'n_classes': n_classes,
            }
            baseline_rows.append(row)
            tag = 'YES' if mirage else 'no'
            print(f'  {arch:>18}: Lab={la:.1f}% Fld={fa:.1f}% '
                  f'DGS_Acc={la-fa:+.2f} DGS_F1={lf-ff:+.2f} '
                  f'BG={bg_delta:.1f} Mirage={tag}')

        results['baseline'] = baseline_rows

        # Multi-seed bootstrap (n_seeds)
        ms_rows = []
        for seed in range(SEED, SEED + n_seeds):
            rng_s = np.random.default_rng(seed)
            li = []
            for c in range(n_classes):
                ci = np.where(all_labels == c)[0]
                n_u = min(n_lab_per_class, int(len(ci) * 0.6))
                li.extend(rng_s.choice(ci, size=n_u, replace=False))
            li = np.array(li)
            ri = np.setdiff1d(np.arange(len(all_labels)), li)

            rl = all_labels[ri]
            av = {c: int((rl == c).sum()) for c in range(n_classes)}
            mN = min(int(av[c] / field_prevalence[c])
                     for c in range(n_classes) if field_prevalence[c] > 0)
            mN = int(mN * 0.9)
            fi_l = []
            for c in range(n_classes):
                ci2 = np.where(rl == c)[0]
                nt = min(int(mN * field_prevalence[c]), len(ci2))
                fi_l.extend(rng_s.choice(ci2, size=nt, replace=False))
            fi = ri[np.array(fi_l)]

            ll = all_labels[li]
            trl2, tel2 = [], []
            for c in range(n_classes):
                cl = np.where(ll == c)[0]
                p = rng_s.permutation(cl)
                s = int(len(p) * 0.8)
                trl2.extend(p[:s]); tel2.extend(p[s:])
            tri = li[np.array(trl2)]
            tei = li[np.array(tel2)]

            for arch in ARCH_NAMES:
                X = all_feats[arch]; y = all_labels
                sc2 = StandardScaler().fit(X[tri])
                clf2 = LogisticRegression(C=1.0, max_iter=1000, random_state=seed)
                clf2.fit(sc2.transform(X[tri]), y[tri])
                la2 = accuracy_score(y[tei], clf2.predict(sc2.transform(X[tei]))) * 100
                lf2 = f1_score(y[tei], clf2.predict(sc2.transform(X[tei])),
                               average='macro') * 100
                fa2 = accuracy_score(y[fi], clf2.predict(sc2.transform(X[fi]))) * 100
                ff2 = f1_score(y[fi], clf2.predict(sc2.transform(X[fi])),
                               average='macro') * 100
                ms_rows.append({
                    'dataset': dataset_name, 'seed': seed, 'arch': arch,
                    'dgs_acc': la2 - fa2, 'dgs_f1': lf2 - ff2,
                    'mirage': (la2 - fa2 < 0) and (lf2 - ff2 > 0),
                })

        results['multiseed'] = ms_rows
        df_ms_cd = pd.DataFrame(ms_rows)
        print(f'\n  Multi-seed ({n_seeds} seeds):')
        for arch in ARCH_NAMES:
            sub = df_ms_cd[df_ms_cd.arch == arch]
            print(f'    {arch:>18}: DGS_Acc={sub.dgs_acc.mean():+.2f}'
                  f'\u00b1{sub.dgs_acc.std():.2f}  '
                  f'DGS_F1={sub.dgs_f1.mean():+.2f}'
                  f'\u00b1{sub.dgs_f1.std():.2f}  '
                  f'Mirage={sub.mirage.sum()}/{len(sub)}')

        # Image-space background (center/border masking)
        bg_sample = rng.choice(field_idx, size=min(1500, len(field_idx)),
                               replace=False)
        bg_paths = all_paths[bg_sample]
        bg_labels = all_labels[bg_sample]
        bg_rows = []
        for arch in ARCH_NAMES:
            model = get_model(arch)
            X_tr = all_feats[arch][train_idx]
            sc3 = StandardScaler().fit(X_tr)
            clf3 = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
            clf3.fit(sc3.transform(X_tr), all_labels[train_idx])

            for cond, mask_fn in [('Original', None),
                                  ('Center-only', center_only_image),
                                  ('Border-only', border_only_image)]:
                if mask_fn is None:
                    f = extract_features(model, bg_paths)
                else:
                    f = extract_masked(model, bg_paths, mask_fn)
                yp = clf3.predict(sc3.transform(f))
                acc = accuracy_score(bg_labels, yp) * 100
                bg_rows.append({'dataset': dataset_name, 'arch': arch,
                                'condition': cond, 'acc': acc})

            del model; torch.cuda.empty_cache()

        results['imagespace'] = bg_rows
        df_bg_cd = pd.DataFrame(bg_rows)
        print(f'\n  Image-space background:')
        for arch in ARCH_NAMES:
            sub = df_bg_cd[df_bg_cd.arch == arch]
            orig = sub[sub.condition == 'Original']['acc'].values[0]
            center = sub[sub.condition == 'Center-only']['acc'].values[0]
            border = sub[sub.condition == 'Border-only']['acc'].values[0]
            print(f'    {arch:>18}: Orig={orig:.1f}% Center={center:.1f}% '
                  f'Border={border:.1f}% '
                  f'BG_reliance={border - 100/n_classes:+.1f}pp')

        return results, pd.DataFrame(baseline_rows), df_ms_cd, df_bg_cd

## Cross-Dataset: PlantVillage Tomato (10 classes)

In [ ]:
if not _SKIP_PART2:
    # Build field prevalence: 65% Early Blight (matches cassava's 65.8% CMD)
    tom_field_prev = {}
    for c in range(N_TOM_CLS):
        if c == tom_eb_idx:
            tom_field_prev[c] = 0.65
        elif c == tom_lb_idx:
            tom_field_prev[c] = 0.12
        elif c == tom_ylcv_idx:
            tom_field_prev[c] = 0.08
        else:
            tom_field_prev[c] = 0.15 / max(N_TOM_CLS - 3, 1)

    # Normalize to sum=1
    s = sum(tom_field_prev.values())
    tom_field_prev = {k: v / s for k, v in tom_field_prev.items()}

    tom_results, df_tom_base, df_tom_ms, df_tom_bg = run_cross_dataset(
        dataset_name='Tomato',
        all_paths=TOM_PATHS, all_labels=TOM_LABELS, class_names=TOM_NAMES,
        all_feats=TOM_FEATS,
        n_lab_per_class=200,
        field_prevalence=tom_field_prev,
        dominant_class_idx=tom_eb_idx,
        dominant_name='Early Blight',
        n_seeds=5,
    )

## Cross-Dataset: Rice Leaf Disease (4 classes)

In [ ]:
if not _SKIP_PART2:
    # Find dominant rice class (bacterial blight or blast)
    rice_dom_idx = 0
    for i, name in enumerate(RICE_NAMES):
        if any(kw in name.lower() for kw in ['blast', 'bacterial', 'blight']):
            rice_dom_idx = i
            break
    print(f'Rice dominant class: [{rice_dom_idx}] {RICE_NAMES[rice_dom_idx]}')

    # 65% dominant, rest split equally
    rice_field_prev = {}
    for c in range(N_RICE_CLS):
        if c == rice_dom_idx:
            rice_field_prev[c] = 0.65
        else:
            rice_field_prev[c] = 0.35 / max(N_RICE_CLS - 1, 1)
    s = sum(rice_field_prev.values())
    rice_field_prev = {k: v / s for k, v in rice_field_prev.items()}

    # Adapt lab size to smallest class
    min_rice = min(int((RICE_LABELS == c).sum()) for c in range(N_RICE_CLS))
    n_lab_rice = min(200, int(min_rice * 0.5))

    rice_results, df_rice_base, df_rice_ms, df_rice_bg = run_cross_dataset(
        dataset_name='Rice',
        all_paths=RICE_PATHS, all_labels=RICE_LABELS, class_names=RICE_NAMES,
        all_feats=RICE_FEATS,
        n_lab_per_class=n_lab_rice,
        field_prevalence=rice_field_prev,
        dominant_class_idx=rice_dom_idx,
        dominant_name=RICE_NAMES[rice_dom_idx],
        n_seeds=5,
    )

## Combined Cross-Dataset Summary (3 crops x 4 architectures)

In [ ]:
if not _SKIP_PART2:
    # Combine all baseline results across 3 datasets
    df_cross_all = pd.concat([df_tom_base, df_rice_base], ignore_index=True)

    # Build cassava baseline reference from Part 1 results
    cassava_rows = []
    for arch in ARCH_NAMES:
        r = BASELINE[arch]
        cassava_rows.append({
            'dataset': 'Cassava', 'arch': arch,
            'lab_acc': r['lab_acc'], 'lab_f1': r['lab_f1'],
            'fld_acc': r['fld_acc'], 'fld_f1': r['fld_f1'],
            'fld_auroc': r['fld_auroc'],
            'dgs_acc': r['dgs_acc'], 'dgs_f1': r['dgs_f1'],
            'bg_delta': r.get('bg_delta', 0), 'mirage': r['mirage'],
            'dominant_pct': (ALL_LABELS[FIELD_IDX] == CMD_IDX).mean() * 100,
            'n_classes': 5,
        })
    df_cassava_base = pd.DataFrame(cassava_rows)
    df_all_datasets = pd.concat([df_cassava_base, df_cross_all], ignore_index=True)

    # Pretty-print
    print('\n' + '=' * 80)
    print('CROSS-DATASET VALIDATION: ALL 4 ARCHITECTURES')
    print('=' * 80)
    hdr = (f'{"Dataset":>10} {"Arch":>18} {"#Cls":>4} {"Dom%":>5} '
           f'{"Lab Acc":>8} {"DGS_Acc":>8} {"DGS_F1":>7} '
           f'{"AUROC":>6} {"BG_d":>6} {"Mirage":>7}')
    print(f'\n{hdr}')
    print('-' * 92)

    for _, r in df_all_datasets.iterrows():
        auroc = r.get('fld_auroc', float('nan'))
        auroc_s = f'{auroc:.1f}' if not np.isnan(auroc) else '---'
        bg = f'{r["bg_delta"]:.1f}' if abs(r.get('bg_delta', 0)) > 0.01 else '---'
        ncls = int(r.get('n_classes', 5))
        tag = 'YES' if r['mirage'] else 'no'
        print(f'{r["dataset"]:>10} {r["arch"]:>18} {ncls:>4} '
              f'{r["dominant_pct"]:>4.0f}% {r["lab_acc"]:>7.1f}% '
              f'{r["dgs_acc"]:>+7.2f} {r["dgs_f1"]:>+6.2f} '
              f'{auroc_s:>6} {bg:>6} {tag:>7}')

    df_all_datasets.to_csv(f'{OUT}/cross_dataset_all_arch.csv', index=False)

    # Mirage summary
    print(f'\nMirage occurrences:')
    for ds in ['Cassava', 'Tomato', 'Rice']:
        sub = df_all_datasets[df_all_datasets.dataset == ds]
        n_mir = sub['mirage'].sum()
        print(f'  {ds}: {n_mir}/{len(sub)} architectures')

    # DGS_F1 vs DGS_Acc magnitude
    print(f'\nDGS_F1 vs DGS_Acc (absolute magnitude):')
    for ds in ['Cassava', 'Tomato', 'Rice']:
        sub = df_all_datasets[df_all_datasets.dataset == ds]
        ratio = (sub['dgs_f1'].abs() / (sub['dgs_acc'].abs() + 1e-8)).mean()
        print(f'  {ds}: mean |DGS_F1|/|DGS_Acc| = {ratio:.1f}x')

In [ ]:
if not _SKIP_PART2:
    # Figure: Cross-dataset DGS summary - 3 panels
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    datasets = ['Cassava', 'Tomato', 'Rice']
    ds_colors = {'Cassava': '#d62728', 'Tomato': '#2ca02c', 'Rice': '#1f77b4'}
    ds_markers = {'Cassava': 'o', 'Tomato': 's', 'Rice': '^'}

    # (a) DGS_Acc vs DGS_F1 scatter
    for ds in datasets:
        sub = df_all_datasets[df_all_datasets.dataset == ds]
        axes[0].scatter(sub['dgs_acc'], sub['dgs_f1'],
                        c=ds_colors[ds], marker=ds_markers[ds],
                        label=ds, s=80, edgecolors='black',
                        linewidths=0.5, alpha=0.85)
    axes[0].axhline(0, color='gray', lw=0.8, ls='--')
    axes[0].axvline(0, color='gray', lw=0.8, ls='--')
    axes[0].fill_between([-15, 0], 0, 30, alpha=0.06, color='red')
    axes[0].set_xlabel(r'DGS$_{\mathrm{Acc}}$ (pp)')
    axes[0].set_ylabel(r'DGS$_{\mathrm{F1}}$ (pp)')
    axes[0].set_title('(a) DGS Across 3 Crops x 4 Architectures')
    axes[0].legend(fontsize=9)
    axes[0].annotate('Mirage\nzone', xy=(-5, 4), fontsize=8,
                     style='italic', color='gray')

    # (b) Image-space: border-only accuracy per dataset
    df_bg_all = pd.concat([
        df_bg.assign(dataset='Cassava'),
        df_tom_bg, df_rice_bg
    ], ignore_index=True)

    y_pos = []
    y_labels = []
    vals = []
    for ds in datasets:
        sub = df_bg_all[df_bg_all.dataset == ds]
        if len(sub) == 0:
            continue
        border_acc = sub[sub.condition == 'Border-only']['acc'].mean()
        y_pos.append(len(y_labels))
        y_labels.append(ds)
        vals.append(border_acc)

    bars = axes[1].barh(y_pos, vals, color=[ds_colors[d] for d in y_labels],
                        alpha=0.85, height=0.5)
    for i, (p, v) in enumerate(zip(y_pos, vals)):
        axes[1].text(v + 0.5, p, f'{v:.1f}%', va='center', fontsize=9)

    # Random-chance lines
    for ds, ncls in [('Cassava', 5), ('Tomato', 10), ('Rice', 4)]:
        if ds in y_labels:
            axes[1].axvline(100 / ncls, color=ds_colors[ds], ls=':', lw=1, alpha=0.5)
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(y_labels)
    axes[1].set_xlabel('Border-Only Accuracy (%)')
    axes[1].set_title('(b) Background-Only Classification\n(dashed = random chance)')

    # (c) Multi-seed DGS_F1 boxplots
    df_ms_all = pd.concat([
        df_ms.assign(dataset='Cassava'),
        df_tom_ms, df_rice_ms
    ], ignore_index=True)

    box_data = []
    box_labels = []
    for ds in datasets:
        sub = df_ms_all[df_ms_all.dataset == ds]
        if len(sub) > 0:
            box_data.append(sub['dgs_f1'].values)
            box_labels.append(ds)

    bp = axes[2].boxplot(box_data, positions=range(len(box_labels)),
                         patch_artist=True, widths=0.6)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(ds_colors[box_labels[i]])
        patch.set_alpha(0.7)
    axes[2].axhline(0, color='black', lw=1, ls='--')
    axes[2].set_xticks(range(len(box_labels)))
    axes[2].set_xticklabels(box_labels)
    axes[2].set_ylabel(r'DGS$_{\mathrm{F1}}$ (pp)')
    axes[2].set_title('(c) Multi-Seed DGS$_{\mathrm{F1}}$ Distribution\n(all architectures pooled)')

    plt.tight_layout()
    plt.savefig(f'{OUT}/fig_cross_dataset_all.pdf', bbox_inches='tight')
    plt.savefig(f'{OUT}/fig_cross_dataset_all.png', bbox_inches='tight', dpi=200)
    plt.show()
    print('Saved: fig_cross_dataset_all.pdf/.png')

        # === Save Part 2 checkpoint (skip tomato+rice re-run) ===
    import pickle as _pkl2
    _ckpt2_data = {
        'TOM_FEATS': TOM_FEATS, 'TOM_LABELS': TOM_LABELS, 'TOM_NAMES': TOM_NAMES,
        'TOM_PATHS': TOM_PATHS, 'tom_field_prev': tom_field_prev,
        'RICE_FEATS': RICE_FEATS, 'RICE_LABELS': RICE_LABELS, 'RICE_NAMES': RICE_NAMES,
        'RICE_PATHS': RICE_PATHS, 'rice_field_prev': rice_field_prev,
        'n_lab_rice': n_lab_rice,
        'df_tom_base': df_tom_base, 'df_rice_base': df_rice_base,
        'df_cassava_base': df_cassava_base if 'df_cassava_base' in dir() else pd.DataFrame(),
        'df_all_datasets': df_all_datasets,
        'tom_results': tom_results, 'rice_results': rice_results,
        'df_tom_ms': df_tom_ms, 'df_tom_bg': df_tom_bg,
        'df_rice_ms': df_rice_ms, 'df_rice_bg': df_rice_bg,
        'N_TOM_CLS': N_TOM_CLS, 'N_RICE_CLS': N_RICE_CLS,
        'tom_eb_idx': tom_eb_idx, 'rice_dom_idx': rice_dom_idx,
    }
    with open(f'{OUT}/part2_checkpoint.pkl', 'wb') as _f2:
        _pkl2.dump(_ckpt2_data, _f2)
    del _ckpt2_data
    print('Part 2 checkpoint saved.')


### Cross-Dataset AUROC/PR Curves + Threshold Sensitivity

In [ ]:
# === CHECKPOINT: Reload Part 2 if available (skip tomato+rice re-run) ===
import pickle as _pkl2
_ckpt2_path = _find_checkpoint('part2_checkpoint.pkl')
_SKIP_PART2 = _ckpt2_path is not None

if _SKIP_PART2:
    print(f'Loading Part 2 checkpoint from: {_ckpt2_path}')
    with open(_ckpt2_path, 'rb') as _f2:
        _ckpt2 = _pkl2.load(_f2)
    TOM_FEATS = _ckpt2['TOM_FEATS']
    TOM_LABELS = _ckpt2['TOM_LABELS']
    TOM_NAMES = _ckpt2['TOM_NAMES']
    TOM_PATHS = _ckpt2['TOM_PATHS']
    tom_field_prev = _ckpt2['tom_field_prev']
    RICE_FEATS = _ckpt2['RICE_FEATS']
    RICE_LABELS = _ckpt2['RICE_LABELS']
    RICE_NAMES = _ckpt2['RICE_NAMES']
    RICE_PATHS = _ckpt2['RICE_PATHS']
    rice_field_prev = _ckpt2['rice_field_prev']
    n_lab_rice = _ckpt2['n_lab_rice']
    df_tom_base = _ckpt2['df_tom_base']
    df_rice_base = _ckpt2['df_rice_base']
    df_cassava_base = _ckpt2['df_cassava_base']
    df_all_datasets = _ckpt2['df_all_datasets']
    tom_results = _ckpt2['tom_results']
    rice_results = _ckpt2['rice_results']
    df_tom_ms = _ckpt2['df_tom_ms']
    df_tom_bg = _ckpt2['df_tom_bg']
    df_rice_ms = _ckpt2['df_rice_ms']
    df_rice_bg = _ckpt2['df_rice_bg']
    N_TOM_CLS = _ckpt2['N_TOM_CLS']
    N_RICE_CLS = _ckpt2['N_RICE_CLS']
    tom_eb_idx = _ckpt2['tom_eb_idx']
    rice_dom_idx = _ckpt2['rice_dom_idx']
    del _ckpt2
    print(f'Loaded Part 2: Tomato {len(TOM_LABELS)} images, Rice {len(RICE_LABELS)} images')
    print(f'  DataFrames: {len(df_all_datasets)} baseline rows, {len(df_tom_ms)} tomato ms, {len(df_rice_ms)} rice ms')
else:
    print('No Part 2 checkpoint - Part 2 cells will run normally.')

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc, roc_auc_score

# AUROC + PR curves for tomato and rice (ResNet-50 representative)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for col, (ds_name, ds_feats, ds_labels, ds_names, fp, n_lab) in enumerate([
    ('Tomato', TOM_FEATS, TOM_LABELS, TOM_NAMES, tom_field_prev, 200),
    ('Rice', RICE_FEATS, RICE_LABELS, RICE_NAMES, rice_field_prev, n_lab_rice),
]):
    n_cls = len(ds_names)
    # Re-derive canonical split (same seed as run_cross_dataset)
    rng_c = np.random.default_rng(SEED)
    li = []
    for c in range(n_cls):
        ci = np.where(ds_labels == c)[0]
        n_u = min(n_lab, int(len(ci) * 0.6))
        li.extend(rng_c.choice(ci, size=n_u, replace=False))
    li = np.array(li)
    ri = np.setdiff1d(np.arange(len(ds_labels)), li)

    rl = ds_labels[ri]
    av = {c: int((rl == c).sum()) for c in range(n_cls)}
    mN = min(int(av[c] / fp[c]) for c in range(n_cls) if fp[c] > 0)
    mN = int(mN * 0.9)
    fi_l = []
    for c in range(n_cls):
        ci = np.where(rl == c)[0]
        nt = min(int(mN * fp[c]), len(ci))
        fi_l.extend(rng_c.choice(ci, size=nt, replace=False))
    fi = ri[np.array(fi_l)]

    ll = ds_labels[li]
    trl, tel = [], []
    for c in range(n_cls):
        cl = np.where(ll == c)[0]
        p = rng_c.permutation(cl)
        s = int(len(p) * 0.8)
        trl.extend(p[:s]); tel.extend(p[s:])
    tri = li[np.array(trl)]
    tei = li[np.array(tel)]

    arch = 'ResNet-50'
    X = ds_feats[arch]
    sc = StandardScaler().fit(X[tri])
    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    clf.fit(sc.transform(X[tri]), ds_labels[tri])

    # Lab ROC + Field ROC
    for row_i, (eval_name, eval_idx) in enumerate([('Lab (test)', tei), ('Field', fi)]):
        y_true = ds_labels[eval_idx]
        y_prob = clf.predict_proba(sc.transform(X[eval_idx]))
        y_bin = label_binarize(y_true, classes=list(range(n_cls)))
        ax = axes[row_i, col]

        for c in range(n_cls):
            if y_bin[:, c].sum() > 0 and y_bin[:, c].sum() < len(y_bin):
                fpr, tpr, _ = roc_curve(y_bin[:, c], y_prob[:, c])
                auc_val = auc(fpr, tpr)
                short_name = ds_names[c][:15]
                ax.plot(fpr, tpr, lw=1.2, alpha=0.7,
                        label=f'{short_name} ({auc_val:.2f})')

        ax.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.4)
        macro_auroc = roc_auc_score(y_bin, y_prob,
                                    multi_class='ovr', average='macro')
        ax.set_title(f'{ds_name} {eval_name}\n'
                     f'Macro AUROC = {macro_auroc:.3f}', fontsize=10)
        ax.set_xlabel('FPR')
        ax.set_ylabel('TPR')
        ax.legend(fontsize=6, loc='lower right', ncol=2)

plt.tight_layout()
plt.savefig(f'{OUT}/fig_cross_dataset_auroc.pdf', bbox_inches='tight')
plt.savefig(f'{OUT}/fig_cross_dataset_auroc.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved: fig_cross_dataset_auroc.pdf/.png')

In [ ]:
# Threshold sensitivity across ALL 3 datasets x 4 architectures
THRESHOLDS = [0.01, 0.03, 0.05, 0.1, 0.15, 0.2]

thresh_all = []
for ds_name, ds_base in [
    ('Cassava', df_cassava_base),
    ('Tomato', df_tom_base),
    ('Rice', df_rice_base),
]:
    for _, brow in ds_base.iterrows():
        for t in THRESHOLDS:
            flagged = abs(brow['dgs_acc']) >= t * 100
            thresh_all.append({
                'dataset': ds_name, 'arch': brow['arch'],
                'threshold': t, 'flagged': flagged,
                'dgs_acc': brow['dgs_acc'], 'mirage': brow['mirage'],
            })

df_thresh = pd.DataFrame(thresh_all)

print('Threshold sensitivity (fraction flagged at each DGS threshold):')
print(f'\n{"Threshold":>10}', end='')
for ds in ['Cassava', 'Tomato', 'Rice']:
    print(f'  {ds:>10}', end='')
print()
print('-' * 45)
for t in THRESHOLDS:
    print(f'{t:>10.0%}', end='')
    for ds in ['Cassava', 'Tomato', 'Rice']:
        sub = df_thresh[(df_thresh.dataset == ds) & (df_thresh.threshold == t)]
        frac = sub['flagged'].mean()
        print(f'  {frac:>10.0%}', end='')
    print()

print('\nRecommended threshold: 5% flags all cassava mirages '
      'while avoiding false alarms on high-accuracy datasets.')

### Theoretical Mirage Inequality Validation (All 3 Datasets)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Validate p1* threshold inequality on all 3 datasets
print('Theoretical mirage inequality: p1* = (Acc_L - r_bar_neg_f) / (r1_f - r_bar_neg_f)')
print('Mirage occurs when field dominant prevalence p1 > p1*\n')

# Cassava uses pre-existing splits from Part 1
for arch in ARCH_NAMES:
    X = ALL_FEATS[arch]; y = ALL_LABELS
    sc = StandardScaler().fit(X[TRAIN_IDX])
    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    clf.fit(sc.transform(X[TRAIN_IDX]), y[TRAIN_IDX])

    acc_lab = accuracy_score(y[TEST_IDX], clf.predict(sc.transform(X[TEST_IDX])))
    y_fld_p = clf.predict(sc.transform(X[FIELD_IDX]))
    y_fld = y[FIELD_IDX]

    r1f = (y_fld_p[y_fld == CMD_IDX] == CMD_IDX).mean()
    mask_neg = y_fld != CMD_IDX
    r_bar_neg = (y_fld_p[mask_neg] == CMD_IDX).mean()

    denom = r1f - r_bar_neg
    if abs(denom) < 1e-8:
        print(f'  Cassava {arch}: degenerate (r1f == r_bar_neg)')
        continue
    p1_star = (acc_lab - r_bar_neg) / denom
    p1_actual = (y_fld == CMD_IDX).mean()
    pred_mirage = p1_actual > p1_star

    la = accuracy_score(y[TEST_IDX], clf.predict(sc.transform(X[TEST_IDX]))) * 100
    fa = accuracy_score(y_fld, y_fld_p) * 100
    lf = f1_score(y[TEST_IDX], clf.predict(sc.transform(X[TEST_IDX])), average='macro') * 100
    ff = f1_score(y_fld, y_fld_p, average='macro') * 100
    actual_mirage = (la - fa < 0) and (lf - ff > 0)
    match = pred_mirage == actual_mirage
    tag_p = 'YES' if pred_mirage else 'no'
    tag_a = 'YES' if actual_mirage else 'no'
    tag_m = 'MATCH' if match else 'MISMATCH'
    print(f'  {"Cassava":>8} {arch:>18}: p1*={p1_star:.3f} p1={p1_actual:.3f} '
          f'Pred={tag_p} Actual={tag_a} {tag_m}')

# Tomato + Rice: re-derive splits with same seed
for ds_name, ds_feats, ds_labels, ds_names, fp, dom_idx, n_lab in [
    ('Tomato', TOM_FEATS, TOM_LABELS, TOM_NAMES, tom_field_prev, tom_eb_idx, 200),
    ('Rice', RICE_FEATS, RICE_LABELS, RICE_NAMES, rice_field_prev, rice_dom_idx, n_lab_rice),
]:
    n_cls = len(ds_names)
    rng_v = np.random.default_rng(SEED)
    li = []
    for c in range(n_cls):
        ci = np.where(ds_labels == c)[0]
        n_u = min(n_lab, int(len(ci) * 0.6))
        li.extend(rng_v.choice(ci, size=n_u, replace=False))
    li = np.array(li)
    ri = np.setdiff1d(np.arange(len(ds_labels)), li)

    rl = ds_labels[ri]
    av = {c: int((rl == c).sum()) for c in range(n_cls)}
    mN = min(int(av[c] / fp[c]) for c in range(n_cls) if fp[c] > 0)
    mN = int(mN * 0.9)
    fi_l = []
    for c in range(n_cls):
        ci = np.where(rl == c)[0]
        nt = min(int(mN * fp[c]), len(ci))
        fi_l.extend(rng_v.choice(ci, size=nt, replace=False))
    fi = ri[np.array(fi_l)]

    ll = ds_labels[li]
    trl, tel = [], []
    for c in range(n_cls):
        cl = np.where(ll == c)[0]
        p = rng_v.permutation(cl)
        s = int(len(p) * 0.8)
        trl.extend(p[:s]); tel.extend(p[s:])
    tri = li[np.array(trl)]
    tei = li[np.array(tel)]

    for arch in ARCH_NAMES:
        X = ds_feats[arch]; y = ds_labels
        sc = StandardScaler().fit(X[tri])
        clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
        clf.fit(sc.transform(X[tri]), y[tri])

        acc_lab = accuracy_score(y[tei], clf.predict(sc.transform(X[tei])))
        y_fld_p = clf.predict(sc.transform(X[fi]))
        y_fld = y[fi]

        r1f = (y_fld_p[y_fld == dom_idx] == dom_idx).mean()
        mask_neg = y_fld != dom_idx
        r_bar_neg = (y_fld_p[mask_neg] == dom_idx).mean()

        denom = r1f - r_bar_neg
        if abs(denom) < 1e-8:
            print(f'  {ds_name} {arch}: degenerate')
            continue
        p1_star = (acc_lab - r_bar_neg) / denom
        p1_actual = (y_fld == dom_idx).mean()
        pred_mirage = p1_actual > p1_star

        la = accuracy_score(y[tei], clf.predict(sc.transform(X[tei]))) * 100
        fa = accuracy_score(y_fld, y_fld_p) * 100
        lf = f1_score(y[tei], clf.predict(sc.transform(X[tei])), average='macro') * 100
        ff = f1_score(y_fld, y_fld_p, average='macro') * 100
        actual_mirage = (la - fa < 0) and (lf - ff > 0)
        match = pred_mirage == actual_mirage
        tag_p = 'YES' if pred_mirage else 'no'
        tag_a = 'YES' if actual_mirage else 'no'
        tag_m = 'MATCH' if match else 'MISMATCH'
        print(f'  {ds_name:>8} {arch:>18}: p1*={p1_star:.3f} p1={p1_actual:.3f} '
              f'Pred={tag_p} Actual={tag_a} {tag_m}')

### LaTeX: Cross-Dataset Table (All 4 Architectures)

In [ ]:
# LaTeX table for paper appendix
print('% Cross-dataset validation table - all 4 architectures + all 5 fixes')
print(r'\begin{table*}[t]')
print(r'\centering')
print(r'\caption{Cross-dataset validation with all four architectures. '
      r'The Evaluation Mirage requires both severe class imbalance '
      r'($>$40\%) \emph{and} moderate model uncertainty '
      r'(lab accuracy $<$75\%). '
      r'DGS$_{\text{F1}}$ exceeds DGS$_{\text{Acc}}$ on all datasets '
      r'regardless of mirage presence.}')
print(r'\label{tab:cross_all}')
print(r'\setlength{\tabcolsep}{3pt}')
print(r'\begin{tabular}{llrrrrrrrr}')
print(r'\toprule')
print(r'\textbf{Dataset} & \textbf{Arch.} & \textbf{\#Cls} & '
      r'\textbf{Dom.\%} & \textbf{Lab Acc} & '
      r'\textbf{DGS$_{\text{Acc}}$} & \textbf{DGS$_{\text{F1}}$} & '
      r'\textbf{AUROC$_F$} & \textbf{BG $\Delta$} & '
      r'\textbf{Mirage} \\')
print(r'\midrule')

prev_ds = None
for _, r in df_all_datasets.iterrows():
    ds = r['dataset']
    if prev_ds is not None and ds != prev_ds:
        print(r'\midrule')
    prev_ds = ds
    aname = (r['arch']
             .replace('EfficientNet-B3', 'EffNet-B3')
             .replace('MobileNetV3-S', 'MobNetV3-S'))
    auroc = r.get('fld_auroc', float('nan'))
    auroc_s = f'{auroc:.1f}' if not np.isnan(auroc) else '---'
    bg = f'{r["bg_delta"]:.1f}' if abs(r.get('bg_delta', 0)) > 0.01 else '---'
    m = 'Yes' if r['mirage'] else 'No'
    ncls = int(r.get('n_classes', 5))
    print(f'{ds} & {aname} & {ncls} & {r["dominant_pct"]:.0f}\\% & '
          f'{r["lab_acc"]:.1f} & ${r["dgs_acc"]:+.1f}$ & '
          f'${r["dgs_f1"]:+.1f}$ & {auroc_s} & {bg} & {m} \\\\')

print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table*}')

In [ ]:
# Save combined cross-dataset results
cross_json = {
    'tomato': {k: v for k, v in tom_results.items() if k != 'imagespace'},
    'rice': {k: v for k, v in rice_results.items() if k != 'imagespace'},
    'combined_baseline': df_all_datasets.to_dict(orient='records'),
}
with open(f'{OUT}/cross_dataset_strengthened.json', 'w') as f:
    json.dump(cross_json, f, indent=2, default=str)

n_base = len(df_all_datasets)
n_ms = len(pd.concat([df_ms.assign(dataset='Cassava'), df_tom_ms, df_rice_ms]))
n_bg = len(pd.concat([df_bg.assign(dataset='Cassava'), df_tom_bg, df_rice_bg]))

print(f'\nAll cross-dataset results saved to {OUT}/')
print(f'Total experiments: {n_base} baseline + {n_ms} multi-seed + {n_bg} image-space')
print(f'= {n_base + n_ms + n_bg} total evaluations across 3 crops x 4 architectures')
print('\nNotebook complete. Upload to Kaggle with GPU T4 x 2 enabled.')